# Debugging a prediction step by step

The `Model` class in `utils.py` has no `predict` method. The only XGBoost call is `booster.predict(..., output_margin=True)` inside `Model._margin`. Everything the project reports is arithmetic on that one number per drone: `risk`, cumulative hazard, survival, conditional survival, hazard, remaining life and every curve of the dashboard.

This notebook redoes that arithmetic by hand for one drone, one named intermediate at a time, and compares each step with the library code. It is meant for debugging: when a number or a curve looks wrong, find the step where the hand-made value and the library value part.

**Roadmap**

| Sections | What is traced | Code under test |
| --- | --- | --- |
| 1 | setup, debug helpers, model state, the drone row | `Model.load` |
| 2 | raw row → feature frame → `DMatrix` → margin; the margin split by tree and by feature | `_features_frame`, `Model._margin`, `Model._predict_margin`, `Model.explain` |
| 3–7 | margin → risk → cumulative hazard → survival → conditional survival → hazard, for the production AFT model | `Model.risk`, `_cum_hazard_at`, `_survival_at`, `survival`, `conditional_survival`, `hazard` |
| 8 | the observation record used by the metrics | `_surv` |
| 9 | the same row through the Cox model | Breslow baseline, kernel-smoothed hazard, frailty link |
| 10 | the whole test table: batch shapes, stored predictions, metrics | `evaluate`, `artifacts/predictions.parquet`, `artifacts/metrics.parquet` |
| 11–15 | which arrays each dashboard figure draws | `dashboard.common`, `dashboard.figures` |
| 16 | index of every intermediate captured along the way | `TRACE` |

**Conventions**

- Shapes: `n` rows (drones), `m` evaluation times, `p` features, `T` trees used for prediction, `k` distinct failure times, `g` grid points of a figure. Per-drone curves are always `(n, m)`: one row per drone, one column per time. Here `n = 1` until section 10.
- `peek(name, obj)` prints type, shape, dtype, number of elements, memory and a preview, and stores the object in `TRACE[name]`.
- `same(name, got, expected)` prints the shape and the largest absolute difference, and raises if shapes or values disagree.
- Every intermediate is also a top-level variable with a unique name, so any of them can be inspected in a new cell after the run.

The walk follows the **production** models (`models/production_*.json`): refitted on train + validation in section 8 of `drone_survival.ipynb`, they score new drones and draw the drone card. `forecast_name` in `artifacts/meta.json` names the one behind the maintenance forecast (section 14). The evaluation models (`models/best_*.json`) appear only where the reported test metrics and the fleet hazard figure are recomputed (sections 10 and 15). The main walk (sections 2–8) is written for AFT; section 9 repeats it for Cox, which takes different branches in `_cum_hazard_at`, `hazard` and `conditional_survival`. To debug another drone, change `DRONE_ID` in section 1.4 and rerun.

`models/`, `artifacts/` and `data/` are only read.

## 1. Setup

### 1.1 Imports and display options

The `utils` logger is routed to the cell output at `WARNING` level, so the warnings the library logs (an unseen category in section 2.1, the flat Cox baseline in section 9.5) appear next to the call that caused them.

In [1]:
import dataclasses
import json
import logging
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xgboost as xgb
from IPython.display import display
from scipy.special import expit
from scipy.stats import norm

from dashboard.common import Settings, drone_hazard_grid, hazard_table
from dashboard.data import DashboardData
from dashboard.figures import conditional_curves, drone_hazard, drone_survival
from dashboard.figures import hazard as fleet_hazard
from dashboard.labels import HAZARD_SCALE, MODEL_LABELS
from utils import (
    SCALAR_METRICS,
    Model,
    _aft_noise,
    _epanechnikov,
    _features_frame,
    _frailty_link,
    _surv,
    encode_unknown_categories,
    evaluate,
    hazard_bandwidth,
    observed_hazard,
    population_hazard,
    remaining_life,
)

log = logging.getLogger("utils")
log.setLevel(logging.WARNING)
log.propagate = False
if not log.handlers:
    log.addHandler(logging.StreamHandler(sys.stdout))

pd.set_option("display.float_format", lambda v: f"{v:.6g}")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

### 1.2 Helpers for inspecting arrays

Three helpers are used in every section below:

- `peek(name, obj)` prints what the object is (type, shape, dtype, number of elements, bytes), the value range of a numeric array and a short preview. It also registers the object in the `TRACE` dictionary under `name`. Names carry the section number, for example `TRACE["4.z"]`.
- `same(name, got, expected)` is the comparison behind every check. Shapes must be equal, with no silent broadcasting, and values must agree to `rtol=1e-10` unless a looser tolerance is passed for `float32` data.
- `trace_index()` returns a table of everything registered so far; section 16 prints it in full.

In [2]:
TRACE: dict[str, object] = {}
_SCALARS = (str, int, float, bool, type(None), np.generic)
_CONTAINERS = (dict, list, tuple, set)


def facts(obj: object) -> dict[str, object]:
    """Type, shape, dtype, element count and memory of ``obj``; ``None`` where undefined."""
    row: dict[str, object] = dict.fromkeys(("type", "shape", "dtype", "size", "bytes"))
    row["type"] = type(obj).__name__
    if isinstance(obj, np.ndarray):
        row.update(shape=obj.shape, dtype=str(obj.dtype), size=obj.size, bytes=obj.nbytes)
    elif isinstance(obj, pd.DataFrame):
        counts = obj.dtypes.astype(str).value_counts()
        row.update(
            shape=obj.shape,
            dtype=", ".join(f"{n} x {name}" for name, n in counts.items()),
            size=obj.size,
            bytes=int(obj.memory_usage(deep=True).sum()),
        )
    elif isinstance(obj, pd.Series):
        row.update(
            shape=obj.shape,
            dtype=str(obj.dtype),
            size=obj.size,
            bytes=int(obj.memory_usage(deep=True)),
        )
    elif isinstance(obj, xgb.DMatrix):
        row.update(shape=(obj.num_row(), obj.num_col()), size=obj.num_row() * obj.num_col())
    elif isinstance(obj, (*_CONTAINERS, str)):
        row.update(size=len(obj))
    return row


def _facts_line(obj: object) -> str:
    """``facts`` as one line, skipping the undefined entries."""
    row = facts(obj)
    keys = ("shape", "dtype", "size", "bytes")
    return " ".join([str(row["type"]), *(f"{key}={row[key]}" for key in keys if row[key] is not None)])


def _brief(obj: object) -> str:
    """The value itself when it is a short builtin, otherwise its type and shape."""
    if isinstance(obj, (*_SCALARS, *_CONTAINERS)) and len(repr(obj)) <= 80:
        return repr(obj)
    return _facts_line(obj)


def _stats(obj: object) -> str:
    """Value range of a numeric array or Series; empty for everything else."""
    values = obj.to_numpy() if isinstance(obj, pd.Series) else obj
    if not isinstance(values, np.ndarray) or values.size == 0:
        return ""
    if values.dtype.kind == "b":
        return f"true={int(values.sum())}"
    if values.dtype.kind not in "iuf":
        return ""
    finite = values[np.isfinite(values)]
    span = f"min={finite.min():.6g} max={finite.max():.6g}" if finite.size else "no finite values"
    if values.dtype.kind != "f":
        return span
    return f"{span} nan={int(np.isnan(values).sum())} inf={int(np.isinf(values).sum())}"


def _preview(obj: object, limit: int) -> str:
    """Short text preview: array edges, frame columns, dictionary entries or sequence items."""
    if isinstance(obj, pd.Series):
        obj = obj.to_numpy()
    if isinstance(obj, np.ndarray):
        return np.array2string(
            obj, threshold=2 * limit, edgeitems=max(limit // 2, 1), precision=6, max_line_width=100
        )
    if isinstance(obj, pd.DataFrame):
        return textwrap.fill("columns: " + ", ".join(map(str, obj.columns)), width=100)
    if isinstance(obj, dict):
        lines = [f"{key!r}: {_brief(value)}" for key, value in list(obj.items())[:limit]]
    elif isinstance(obj, (list, tuple)) and all(isinstance(item, _SCALARS) for item in obj):
        lines = [textwrap.fill(repr(list(obj[:limit])), width=100)]
    elif isinstance(obj, (list, tuple)):
        lines = [f"[{i}] {_brief(item)}" for i, item in enumerate(obj[:limit])]
    else:
        return ""
    if len(obj) > limit:
        lines.append(f"... {len(obj) - limit} more")
    return "\n".join(lines)


def peek(name: str, obj: object, limit: int = 6) -> None:
    """Register ``obj`` in ``TRACE[name]`` and print its type, shape, dtype, size and a preview.

    Parameters
    ----------
    name : str
        Key in ``TRACE``; by convention ``"<section>.<variable>"``.
    obj : object
        Array, frame, container, scalar or any other intermediate result.
    limit : int, default 6
        Number of container items shown; arrays longer than ``2 * limit`` show their edges.
    """
    TRACE[name] = obj
    if isinstance(obj, _SCALARS):
        print(f"[{name}] {type(obj).__name__} = {obj!r}")
        return
    stats = _stats(obj)
    print(f"[{name}] {_facts_line(obj)}" + (f" | {stats}" if stats else ""))
    for line in _preview(obj, limit).splitlines():
        print(f"    {line}")


def same(
    name: str, got: object, expected: object, rtol: float = 1e-10, atol: float = 1e-12
) -> None:
    """Check that ``got`` equals ``expected`` in shape and value and print the largest gap.

    Parameters
    ----------
    name : str
        Label of the check.
    got, expected : array-like
        Values to compare; converted to ``float`` arrays.
    rtol, atol : float
        Tolerances of ``numpy.testing.assert_allclose``.

    Raises
    ------
    AssertionError
        If the shapes differ or any element is outside the tolerance.
    """
    got = np.asarray(got, dtype=float)
    expected = np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        raise AssertionError(f"{name}: shape {got.shape} != {expected.shape}")
    with np.errstate(invalid="ignore"):
        gap = np.abs(got - expected)
    finite = gap[np.isfinite(gap)]
    diff = float(finite.max()) if finite.size else 0.0
    try:
        np.testing.assert_allclose(got, expected, rtol=rtol, atol=atol)
    except AssertionError:
        print(f"[MISMATCH] {name}: shape={got.shape}, max |diff| = {diff:.3e}")
        raise
    print(f"[ok] {name}: shape={got.shape}, max |diff| = {diff:.3e}")


def trace_index() -> pd.DataFrame:
    """Everything registered in ``TRACE`` so far: name, type, shape, dtype, elements, bytes."""
    table = pd.DataFrame([{"name": name, **facts(obj)} for name, obj in TRACE.items()])
    return table.astype({"size": "Int64", "bytes": "Int64"})

### 1.3 Helpers for inspecting figures

A Plotly figure is a container of arrays. `fig.data` is a tuple of traces, each with its own `x` and `y`; `fig.layout.shapes` holds reference lines and `fig.layout.annotations` their labels. Debugging a plot means checking that these arrays are the ones you expect.

- `figure_traces(fig)` returns one row per trace: name, drawing mode, array shapes and value ranges.
- `figure_marks(fig)` returns the reference lines and annotations with their coordinates. A coordinate reference such as `y domain` means the value is a fraction of the axis length, not a data value.
- `trace_xy(fig, name)` returns the plotted `x` and `y` of one trace as a frame; unnamed traces are selected by position.
- `describe_figure(name, fig)` registers the figure in `TRACE` and prints all of the above.

In [3]:
def _span(values: np.ndarray) -> tuple[float, float]:
    """Smallest and largest finite value; NaN for an array that has none."""
    finite = values[np.isfinite(values)]
    return (float(finite.min()), float(finite.max())) if finite.size else (np.nan, np.nan)


def figure_traces(fig: go.Figure) -> pd.DataFrame:
    """One row per trace of ``fig``: label, drawing mode, array shapes and value ranges."""
    rows = []
    for i, tr in enumerate(fig.data):
        x = np.asarray(tr.x, dtype=float)
        y = np.asarray(tr.y, dtype=float)
        (x_min, x_max), (y_min, y_max) = _span(x), _span(y)
        rows.append(
            {
                "trace": i,
                "name": tr.name,
                "type": tr.type,
                "mode": tr.mode,
                "x shape": x.shape,
                "y shape": y.shape,
                "x min": x_min,
                "x max": x_max,
                "y min": y_min,
                "y max": y_max,
                "NaN in y": int(np.isnan(y).sum()),
                "line colour": tr.line.color,
                "line width": tr.line.width,
                "fill": tr.fill,
                "in legend": tr.showlegend is not False,
            }
        )
    return pd.DataFrame(rows)


def figure_marks(fig: go.Figure) -> pd.DataFrame:
    """Reference lines (``layout.shapes``) and labels (``layout.annotations``) of ``fig``."""
    shapes = [
        {
            "kind": f"shape/{s.type}",
            "xref": s.xref,
            "x0": s.x0,
            "x1": s.x1,
            "yref": s.yref,
            "y0": s.y0,
            "y1": s.y1,
            "colour": s.line.color,
            "dash": s.line.dash,
            "text": None,
        }
        for s in fig.layout.shapes
    ]
    notes = [
        {
            "kind": "annotation",
            "xref": a.xref,
            "x0": a.x,
            "x1": a.x,
            "yref": a.yref,
            "y0": a.y,
            "y1": a.y,
            "colour": None,
            "dash": None,
            "text": a.text,
        }
        for a in fig.layout.annotations
    ]
    return pd.DataFrame(shapes + notes)


def trace_xy(fig: go.Figure, name: str | int) -> pd.DataFrame:
    """``x`` and ``y`` of one trace, selected by legend name or by position in ``fig.data``.

    Raises
    ------
    RuntimeError
        If ``name`` is a string and does not match exactly one trace.
    """
    if isinstance(name, int):
        tr = fig.data[name]
    else:
        matches = [tr for tr in fig.data if tr.name == name]
        if len(matches) != 1:
            raise RuntimeError(f"{name}: {len(matches)} traces, expected one")
        tr = matches[0]
    return pd.DataFrame({"x": np.asarray(tr.x, dtype=float), "y": np.asarray(tr.y, dtype=float)})


def describe_figure(name: str, fig: go.Figure) -> None:
    """Register ``fig`` in ``TRACE[name]`` and show its layout, traces and reference marks."""
    TRACE[name] = fig
    layout = fig.layout
    print(
        f"[{name}] Figure: {len(fig.data)} traces, {len(layout.shapes)} shapes, "
        f"{len(layout.annotations)} annotations"
    )
    print(f"    title:  {layout.title.text!r}")
    for axis_name, axis in (("x", layout.xaxis), ("y", layout.yaxis)):
        print(
            f"    {axis_name} axis: {axis.title.text!r}, range={axis.range}, "
            f"rangemode={axis.rangemode}"
        )
    print(f"    height: {layout.height}")
    display(figure_traces(fig))
    marks = figure_marks(fig)
    if len(marks):
        display(marks)

### 1.4 Parameters

`DRONE_ID` selects the row that is traced. `PROBE_HOUR` is an arbitrary early time; the other two probe times are the drone's current age and that age plus `LOOKAHEAD_H`. Three points are enough to print every intermediate in full; sections 12–15 switch to the dashboard grids.

In [4]:
DRONE_ID = "DR-003057"  # any drone_id from data/drone_survival.csv
PROBE_HOUR = 50.0  # an early time, in flight hours
LOOKAHEAD_H = 100.0  # look-ahead added to the drone's age, in flight hours

DATA_PATH = Path("data/drone_survival.csv")
ARTIFACTS_DIR = Path("artifacts")
MODELS_DIR = Path("models")

### 1.5 Run metadata

`artifacts/meta.json` is written by the last cell of `drone_survival.ipynb`. It names the model selected on validation (`best_name`), the production model that makes the forecasts (`forecast_name`) and the model files of both stages, and stores the settings the dashboard needs: the feature list, the category vocabulary, the evaluation times of the metrics, the remaining-life horizon and window. `artifacts/model_card.json` holds the release-gate result and the provenance of the run.

In [5]:
META = json.loads((ARTIFACTS_DIR / "meta.json").read_text())
peek("1.META", META, limit=len(META))
CARD = json.loads((ARTIFACTS_DIR / "model_card.json").read_text())
print(
    f"\nselected on validation: {META['best_name']}; forecasts: production {META['forecast_name']}; "
    f"release gate passed: {[kind for kind, result in CARD['gate']['kinds'].items() if result['passed']]}"
)
print(f"evaluation model files: {META['models']}; production: {META['production_models']}")

[1.META] dict size=21
    'format': 2
    'generated_at': '2026-11-19T11:44:43'
    'best_name': 'aft'
    'forecast_name': 'cox'
    'models': {'cox': 'best_cox.json', 'aft': 'best_aft.json'}
    'production_models': {'cox': 'production_cox.json', 'aft': 'production_aft.json'}
    'features': list size=16
    'cat_features': ['model', 'environment', 'is_refurbished', 'firmware_major']
    'categories': dict size=3
    'numeric_ranges': dict size=12
    'rul_horizon': 1111.0
    'rul_window': 100.0
    'split': dict size=5
    'n_trials': 60
    'n_boot': 1000
    'eval_times': dict size=2
    'calibration_horizons': [118.95, 219.12, 347.5]
    'landmark': 118.95
    'priority_thresholds': {'high': 0.5, 'medium': 0.2}
    'shap_base': {'cox': -0.044754818081855774, 'aft': -3.961398124694824}
    'shap_base_production': {'cox': -0.042453616857528687, 'aft': -3.7449607849121094}

selected on validation: aft; forecasts: production cox; release gate passed: ['cox']
evaluation model files: 

### 1.6 The model object

`Model.load` restores everything a prediction needs, not just the booster:

| Attribute | Content | Used by |
| --- | --- | --- |
| `kind` | `"cox"` or `"aft"` | every branch below |
| `params` | tuned XGBoost parameters; for AFT the noise distribution and its scale $\sigma$ | `_cum_hazard_at`, `hazard` |
| `features`, `cat_features` | columns of the feature frame, in training order | `_features_frame` |
| `categories` | category vocabulary learned on the rows the model was fitted on (train + validation for a production model) | `encode_unknown_categories` |
| `booster` | the trees; `best_iteration` is the last round used: the early-stopping round of an evaluation model, the last stored round of a production model | `_margin` |
| `_breslow` | Cox only: failure times of the fitting rows and the cumulative baseline hazard | `_cum_hazard_at`, `hazard` |
| `calibration` | parameters of the gamma-frailty link, or `None` | `_survival_at`, `hazard` |
| `history` | loss per boosting round; a production model has the training loss only | learning curves only |

For AFT the margin is the predicted $\log T$ and $\sigma$ is the noise scale `aft_loss_distribution_scale`.

In [6]:
MODEL = Model.load(MODELS_DIR / META["production_models"]["aft"])
if MODEL.kind != "aft":
    raise RuntimeError(
        f"The main walk is written for AFT, but production_models['aft'] in meta.json is {MODEL.kind}"
    )

dist = MODEL._aft_distribution
sigma = float(MODEL.params["aft_loss_distribution_scale"])
N_ROUNDS = MODEL.booster.num_boosted_rounds()
BEST_ITERATION = MODEL.booster.best_iteration
N_TREES_USED = BEST_ITERATION + 1
history_lengths = {
    f"{split}/{metric}": len(values)
    for split, metrics in MODEL.history.items()
    for metric, values in metrics.items()
}

peek("1.MODEL.params", MODEL.params, limit=len(MODEL.params))
peek("1.MODEL.features", MODEL.features, limit=len(MODEL.features))
peek("1.MODEL.cat_features", MODEL.cat_features)
peek("1.MODEL.categories", MODEL.categories)
peek("1.MODEL.calibration", MODEL.calibration)
peek("1.MODEL._breslow", MODEL._breslow)
peek("1.MODEL.history_lengths", history_lengths)

print(f"\nmodel: production {MODEL.kind} ({META['production_models']['aft']})")
print(f"noise distribution: {dist}, sigma={sigma:.6g}")
print(
    f"boosting rounds stored: {N_ROUNDS}; best_iteration: {BEST_ITERATION}; "
    f"trees used for prediction: {N_TREES_USED}"
)
print(f"features seen by the booster: {MODEL.booster.num_features()}")

[1.MODEL.params] dict size=8
    'max_depth': 5
    'eta': 0.03159973167182181
    'min_child_weight': 14.606808553399388
    'subsample': 0.9793059603844647
    'colsample_bytree': 0.8022842436370485
    'reg_lambda': 1.618230350203502
    'aft_loss_distribution': 'logistic'
    'aft_loss_distribution_scale': 0.7381798504708477
[1.MODEL.features] list size=16
    ['model', 'environment', 'is_refurbished', 'initial_battery_cycles', 'firmware_major',
    'maintenance_interval_h', 'operator_experience_yrs', 'daily_utilization_h', 'avg_flight_min',
    'avg_payload_kg', 'payload_ratio', 'mean_temp_c', 'avg_humidity_pct', 'avg_wind_ms',
    'hard_landings_per_100h', 'mean_vibration_g']
[1.MODEL.cat_features] list size=4
    ['model', 'environment', 'is_refurbished', 'firmware_major']
[1.MODEL.categories] dict size=3
    'model': ['fixed_wing_vtol', 'hexa_heavy', 'quad_heavy', 'quad_light']
    'environment': ['coastal', 'desert', 'forest', 'urban']
    'firmware_major': [3, 4, 5]
[1.MODEL.

### 1.7 The drone row and the probe times

The row comes from `data/drone_survival.csv` as is: one line per drone with its features, the flight hours observed so far (`duration_h`) and whether the observation ended in a failure (`event = 1`) or the drone was still flying when it was last seen (`event = 0`, right-censored).

`AGE` is `duration_h`. `TIMES` holds the three probe times: `PROBE_HOUR`, `AGE` and `AGE + LOOKAHEAD_H`.

In [7]:
fleet = pd.read_csv(DATA_PATH)
drone = fleet.loc[fleet["drone_id"] == DRONE_ID].reset_index(drop=True)
if len(drone) != 1:
    raise ValueError(f"{DRONE_ID}: expected one row, found {len(drone)}")

AGE = float(drone["duration_h"].iloc[0])
EVENT = int(drone["event"].iloc[0])
TIMES = np.array([PROBE_HOUR, AGE, AGE + LOOKAHEAD_H])

peek("1.fleet", fleet)
peek("1.drone", drone)
peek("1.AGE", AGE)
peek("1.EVENT", EVENT)
peek("1.TIMES", TIMES)
status = "failure observed" if EVENT == 1 else "censored, still flying when last seen"
print(f"\n{DRONE_ID}: {AGE:g} h observed, event={EVENT} ({status})")
drone.T

[1.fleet] DataFrame shape=(5000, 21) dtype=11 x float64, 5 x str, 4 x int64, 1 x bool size=105000 bytes=1021733
    columns: drone_id, model, environment, commission_date, is_refurbished, initial_battery_cycles,
    firmware_major, maintenance_interval_h, operator_experience_yrs, daily_utilization_h,
    avg_flight_min, avg_payload_kg, payload_ratio, mean_temp_c, avg_humidity_pct, avg_wind_ms,
    hard_landings_per_100h, mean_vibration_g, duration_h, event, failure_mode
[1.drone] DataFrame shape=(1, 21) dtype=11 x float64, 5 x str, 4 x int64, 1 x bool size=21 bytes=340
    columns: drone_id, model, environment, commission_date, is_refurbished, initial_battery_cycles,
    firmware_major, maintenance_interval_h, operator_experience_yrs, daily_utilization_h,
    avg_flight_min, avg_payload_kg, payload_ratio, mean_temp_c, avg_humidity_pct, avg_wind_ms,
    hard_landings_per_100h, mean_vibration_g, duration_h, event, failure_mode
[1.AGE] float = 193.5
[1.EVENT] int = 0
[1.TIMES] ndarray sha

,0
drone_id,DR-003057
model,quad_light
environment,urban
commission_date,2025-04-17
is_refurbished,False
initial_battery_cycles,24
firmware_major,3
maintenance_interval_h,200
operator_experience_yrs,1.44
daily_utilization_h,0.75


## 2. From a raw row to the margin

### 2.1 Feature frame

`Model._predict_margin` first calls `_features_frame`, which does three things:

1. keeps only the `features` columns, in training order;
2. casts the categorical columns to the training vocabulary with `encode_unknown_categories`; a level that was not seen in training becomes NaN (category code −1) and a warning is logged;
3. rejects columns whose dtype XGBoost cannot take.

`is_refurbished` is listed in `cat_features` but stays `bool`: `fit_categories` skips boolean columns, so it has no vocabulary. `duration_h`, `event` and the identifiers are dropped here, which is why a prediction cannot depend on them.

The table shows each feature before and after encoding. `category code` is the position of the value in the training vocabulary.

In [8]:
raw_features = drone[MODEL.features]
vocabulary = {col: levels for col, levels in MODEL.categories.items() if col in MODEL.features}
X_hand = encode_unknown_categories(raw_features, vocabulary)
X = _features_frame(drone, MODEL.features, MODEL.categories)
pd.testing.assert_frame_equal(X, X_hand)
print("[ok] _features_frame = column selection + encode_unknown_categories")

dropped = sorted(set(drone.columns) - set(X.columns))
peek("2.raw_features", raw_features)
peek("2.vocabulary", vocabulary)
peek("2.X", X)
peek("2.dropped", dropped)

is_categorical = [isinstance(dtype, pd.CategoricalDtype) for dtype in X.dtypes]
feature_table = pd.DataFrame(
    {
        "raw dtype": raw_features.dtypes.astype(str).to_numpy(),
        "encoded dtype": X.dtypes.astype(str).to_numpy(),
        "value": [X[col].iloc[0] for col in X.columns],
        "category code": pd.array(
            [
                int(X[col].cat.codes.iloc[0]) if categorical else None
                for col, categorical in zip(X.columns, is_categorical, strict=True)
            ],
            dtype="Int64",
        ),
        "vocabulary": [vocabulary.get(col) for col in X.columns],
    },
    index=X.columns,
)
feature_table

[ok] _features_frame = column selection + encode_unknown_categories
[2.raw_features] DataFrame shape=(1, 16) dtype=10 x float64, 3 x int64, 2 x str, 1 x bool size=16 bytes=270
    columns: model, environment, is_refurbished, initial_battery_cycles, firmware_major,
    maintenance_interval_h, operator_experience_yrs, daily_utilization_h, avg_flight_min,
    avg_payload_kg, payload_ratio, mean_temp_c, avg_humidity_pct, avg_wind_ms, hard_landings_per_100h,
    mean_vibration_g
[2.vocabulary] dict size=3
    'model': ['fixed_wing_vtol', 'hexa_heavy', 'quad_heavy', 'quad_light']
    'environment': ['coastal', 'desert', 'forest', 'urban']
    'firmware_major': [3, 4, 5]
[2.X] DataFrame shape=(1, 16) dtype=10 x float64, 3 x category, 2 x int64, 1 x bool size=16 bytes=389
    columns: model, environment, is_refurbished, initial_battery_cycles, firmware_major,
    maintenance_interval_h, operator_experience_yrs, daily_utilization_h, avg_flight_min,
    avg_payload_kg, payload_ratio, mean_temp_c

,raw dtype,encoded dtype,value,category code,vocabulary
model,str,category,quad_light,3,"[fixed_wing_vtol, hexa_heavy, quad_heavy, quad..."
environment,str,category,urban,3,"[coastal, desert, forest, urban]"
is_refurbished,bool,bool,False,<NA>,None
initial_battery_cycles,int64,int64,24,<NA>,None
firmware_major,int64,category,3,0,"[3, 4, 5]"
maintenance_interval_h,int64,int64,200,<NA>,None
operator_experience_yrs,float64,float64,1.44,<NA>,None
daily_utilization_h,float64,float64,0.75,<NA>,None
avg_flight_min,float64,float64,23.5,<NA>,None
avg_payload_kg,float64,float64,0.722,<NA>,None


An unseen level does not stop the prediction. The cell below copies the row, replaces `model` with a level that is not in the vocabulary and encodes it: the warning is logged, the value becomes NaN and the booster still returns a margin. If a new drone gets a surprising prediction, this is the first thing to check.

In [9]:
unseen = drone.copy()
unseen["model"] = "not_in_training"
X_unseen = _features_frame(unseen, MODEL.features, MODEL.categories)
margin_unseen = MODEL._margin(xgb.DMatrix(X_unseen, enable_categorical=True))
peek("2.X_unseen.model", X_unseen["model"])
peek("2.margin_unseen", margin_unseen)
print(f"category code of the unseen level: {int(X_unseen['model'].cat.codes.iloc[0])}")

model: 1 rows with unseen categories ['not_in_training'] set to NaN


[2.X_unseen.model] Series shape=(1,) dtype=category size=1 bytes=210
    [nan]
[2.margin_unseen] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=4.62486 max=4.62486 nan=0 inf=0
    [4.624861]
category code of the unseen level: -1


### 2.2 DMatrix

The encoded frame is wrapped in an `xgb.DMatrix` with `enable_categorical=True` and **no labels**: `duration_h` and `event` are needed for training only. The feature names of the matrix must match what the booster was trained on, and the table lets you compare the types.

XGBoost type codes: `c` categorical, `i` indicator (boolean), `int` integer, `float` real.

In [10]:
dm = xgb.DMatrix(X, enable_categorical=True)
dm_labels = dm.get_label()
peek("2.dm", dm)
peek("2.dm_labels", dm_labels)
if dm_labels.size:
    raise RuntimeError("the prediction DMatrix must not carry labels")
if dm.feature_names != MODEL.booster.feature_names:
    raise RuntimeError("DMatrix and booster disagree on the feature names")
print(f"DMatrix: {dm.num_row()} row(s), {dm.num_col()} columns, {dm_labels.size} labels")
print("columns of the row that did not enter the matrix:", dropped)

pd.DataFrame(
    {
        "pandas dtype": X.dtypes.astype(str).to_numpy(),
        "DMatrix type": dm.feature_types,
        "booster type": MODEL.booster.feature_types,
    },
    index=dm.feature_names,
)

[2.dm] DMatrix shape=(1, 16) size=16
[2.dm_labels] ndarray shape=(0,) dtype=float32 size=0 bytes=0
    []
DMatrix: 1 row(s), 16 columns, 0 labels
columns of the row that did not enter the matrix: ['commission_date', 'drone_id', 'duration_h', 'event', 'failure_mode']


,pandas dtype,DMatrix type,booster type
model,category,c,c
environment,category,c,c
is_refurbished,bool,i,i
initial_battery_cycles,int64,int,int
firmware_major,category,c,c
maintenance_interval_h,int64,int,int
operator_experience_yrs,float64,float,float
daily_utilization_h,float64,float,float
avg_flight_min,float64,float,float
avg_payload_kg,float64,float,float


### 2.3 Margin

`Model._margin` is the only place where XGBoost predicts:

$$\hat f(x) = \texttt{booster.predict}(\texttt{output\_margin=True},\ \texttt{iteration\_range}=(0,\ \texttt{best\_iteration}+1)).$$

Three details matter when debugging:

- `output_margin=True` returns the raw sum of the trees. Without it XGBoost applies $\exp$ for both survival objectives; the cell checks that the result is $\exp(\hat f)$.
- `iteration_range` limits the sum to `best_iteration + 1` trees. An evaluation model (`best_*.json`) also stores the rounds trained after the early-stopping round, so for it a bare `booster.predict(dm, output_margin=True)` is not limited to the best iteration. A production model is trained for exactly that number of rounds, so here the three results coincide; the cell prints the bare result next to the margin of all stored rounds and the margin `Model` uses.
- The result is `float32` with shape `(n,)`: one margin per row.

For AFT $\hat f(x)$ is the predicted $\log T$. $\exp(\hat f)$ is the time at which the standardised log-time $z$ of section 4 is zero; section 5 evaluates $S$ there.

In [11]:
iteration_range = (0, MODEL.booster.best_iteration + 1)
direct = MODEL.booster.predict(dm, output_margin=True, iteration_range=iteration_range)
via_margin = MODEL._margin(dm)
margin = MODEL._predict_margin(drone)
same("booster.predict = _margin", direct, via_margin)
same("_margin = _predict_margin", via_margin, margin)

transformed = MODEL.booster.predict(dm, iteration_range=iteration_range)
same("predict without output_margin = exp(margin)", transformed, np.exp(margin), rtol=1e-6)

margin_all_rounds = MODEL.booster.predict(
    dm, output_margin=True, iteration_range=(0, N_ROUNDS)
)
margin_bare = MODEL.booster.predict(dm, output_margin=True)

peek("2.margin", margin)
peek("2.transformed", transformed)
peek("2.margin_all_rounds", margin_all_rounds)
peek("2.margin_bare", margin_bare)
print(f"\nmargin = predicted log T = {margin[0]:.6f}")
print(f"exp(margin) = {np.exp(margin[0]):.2f} h")
print(
    f"margin of all {N_ROUNDS} rounds = {margin_all_rounds[0]:.6f}; "
    f"bare booster.predict = {margin_bare[0]:.6f}; "
    f"bare call minus the margin Model uses = {margin_bare[0] - margin[0]:+.6f}"
)

[ok] booster.predict = _margin: shape=(1,), max |diff| = 0.000e+00
[ok] _margin = _predict_margin: shape=(1,), max |diff| = 0.000e+00
[ok] predict without output_margin = exp(margin): shape=(1,), max |diff| = 0.000e+00
[2.margin] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=4.57009 max=4.57009 nan=0 inf=0
    [4.570089]
[2.transformed] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=96.5527 max=96.5527 nan=0 inf=0
    [96.55269]
[2.margin_all_rounds] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=4.57009 max=4.57009 nan=0 inf=0
    [4.570089]
[2.margin_bare] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=4.57009 max=4.57009 nan=0 inf=0
    [4.570089]

margin = predicted log T = 4.570089
exp(margin) = 96.55 h
margin of all 346 rounds = 4.570089; bare booster.predict = 4.570089; bare call minus the margin Model uses = +0.000000


### 2.4 The margin tree by tree

The margin is an intercept plus one leaf value per tree:

$$\hat f(x) = b + \sum_{j=0}^{T-1} w_j(x), \qquad T = \texttt{best\_iteration} + 1.$$

- $b$ comes from the booster's `base_score`, read from `booster.save_config()`. Here it enters the margin as $b = \log(\texttt{base\_score})$; the first check of the cell confirms it for the loaded model.
- $w_j(x)$ is the value of the leaf that row $x$ reaches in tree $j$. Predicting with `iteration_range=(j, j + 1)` returns $b + w_j(x)$.
- `pred_leaf=True` returns the index of that leaf in every tree, shape `(n, T)`.
- The staged margin after $j + 1$ trees, `iteration_range=(0, j + 1)`, shows how the prediction converges over the boosting rounds.

XGBoost computes in `float32`, so sums rebuilt here agree with it to about $10^{-6}$, not to machine precision.

In [12]:
booster_config = json.loads(MODEL.booster.save_config())
base_score_raw = booster_config["learner"]["learner_model_param"]["base_score"]
base_score = float(np.atleast_1d(json.loads(base_score_raw))[0])
intercept = float(np.log(base_score))

tree_index = np.arange(N_TREES_USED)
staged_margin = np.array(
    [
        MODEL.booster.predict(dm, output_margin=True, iteration_range=(0, j + 1))[0]
        for j in tree_index
    ],
    dtype=float,
)
single_tree = np.array(
    [
        MODEL.booster.predict(dm, output_margin=True, iteration_range=(j, j + 1))[0]
        for j in tree_index
    ],
    dtype=float,
)
tree_values = single_tree - intercept
leaf_index = MODEL.booster.predict(dm, pred_leaf=True, iteration_range=iteration_range)

peek("2.base_score", base_score)
peek("2.intercept", intercept)
peek("2.staged_margin", staged_margin)
peek("2.single_tree", single_tree)
peek("2.tree_values", tree_values)
peek("2.leaf_index", leaf_index)

same(
    "intercept + cumulative leaf values = staged margin",
    intercept + np.cumsum(tree_values),
    staged_margin,
    rtol=0,
    atol=1e-4,
)
same("staged margin after the last tree = margin", staged_margin[-1:], margin)

tree_table = pd.DataFrame(
    {
        "tree": tree_index,
        "leaf index": leaf_index[0].astype(int),
        "leaf value": tree_values,
        "staged margin": staged_margin,
    }
)
by_size = tree_table["leaf value"].abs().sort_values(ascending=False).index
print("\nfirst and last trees:")
display(pd.concat([tree_table.head(5), tree_table.tail(5)]))
print("trees that move the margin most:")
display(tree_table.loc[by_size].head(5))

[2.base_score] float = 0.5
[2.intercept] float = -0.6931471805599453
[2.staged_margin] ndarray shape=(346,) dtype=float64 size=346 bytes=2768 | min=4.56539 max=6.20783 nan=0 inf=0
    [6.207828 6.193474 6.156084 ... 4.58793  4.565393 4.570089]
[2.single_tree] ndarray shape=(346,) dtype=float64 size=346 bytes=2768 | min=-0.74738 max=6.20783 nan=0 inf=0
    [ 6.207828 -0.7075   -0.730538 ... -0.693142 -0.715684 -0.688451]
[2.tree_values] ndarray shape=(346,) dtype=float64 size=346 bytes=2768 | min=-0.054233 max=6.90097 nan=0 inf=0
    [ 6.900975e+00 -1.435322e-02 -3.739041e-02 ...  5.660537e-06 -2.253658e-02  4.696010e-03]
[2.leaf_index] ndarray shape=(1, 346) dtype=float32 size=346 bytes=1384 | min=1 max=49 nan=0 inf=0
    [[ 1. 22.  8. ... 22. 35.  9.]]
[ok] intercept + cumulative leaf values = staged margin: shape=(346,), max |diff| = 2.281e-06
[ok] staged margin after the last tree = margin: shape=(1,), max |diff| = 0.000e+00

first and last trees:


,tree,leaf index,leaf value,staged margin
0,0,1,6.90097,6.20783
1,1,22,-0.0143532,6.19347
2,2,8,-0.0373904,6.15608
3,3,19,-0.0389404,6.11714
4,4,27,-0.00469047,6.11245
341,341,24,-0.000157358,4.58893
342,342,24,-0.00100857,4.58792
343,343,22,5.66054e-06,4.58793
344,344,35,-0.0225366,4.56539
345,345,9,0.00469601,4.57009


trees that move the margin most:


,tree,leaf index,leaf value,staged margin
0,0,1,6.90097,6.20783
6,6,39,-0.054233,6.0228
9,9,13,-0.0474762,5.95766
3,3,19,-0.0389404,6.11714
11,11,15,-0.0376016,5.89416


### 2.5 The margin feature by feature

`pred_contribs=True` returns TreeSHAP contributions with shape `(n, p + 1)`: one column per feature plus a last bias column. Each row sums to the margin:

$$\hat f(x) = \phi_0 + \sum_{i=1}^{p} \phi_i(x).$$

The bias $\phi_0$ is the SHAP base value, the margin expected before any feature is known. It is **not** the intercept $b$ of section 2.4; the cell prints both.

`Model.explain` returns the same numbers for the `risk` score, so for AFT the signs are flipped (section 3): a positive value pushes towards earlier failure. Its base value is what `meta.json` stores as `shap_base`, the starting point of the dashboard waterfall.

In [13]:
contribs = MODEL.booster.predict(dm, pred_contribs=True, iteration_range=iteration_range)
feature_contribs = contribs[:, :-1]
bias = contribs[:, -1]
peek("2.contribs", contribs)
peek("2.feature_contribs", feature_contribs)
peek("2.bias", bias)
same("bias + sum of contributions = margin", contribs.sum(axis=1), margin, rtol=0, atol=1e-4)

explanation = MODEL.explain(drone)
peek("2.explanation.values", explanation.values)
peek("2.explanation.base_values", explanation.base_values)
same("explain values = -contributions", explanation.values, -feature_contribs, rtol=0, atol=1e-5)
same("explain base value = -bias", explanation.base_values, -bias, rtol=0, atol=1e-5)
same(
    "explain base value = shap_base in meta.json",
    explanation.base_values,
    np.array([META["shap_base_production"][MODEL.kind]]),
    rtol=0,
    atol=1e-5,
)
print(f"\nintercept b = {intercept:.6f}; SHAP bias = {bias[0]:.6f}")

contrib_table = pd.DataFrame(
    {
        "value": [X[col].iloc[0] for col in X.columns],
        "contribution to margin": feature_contribs[0],
        "contribution to risk": explanation.values[0],
    },
    index=X.columns,
)
by_size = contrib_table["contribution to margin"].abs().sort_values(ascending=False).index
contrib_table.loc[by_size]

[2.contribs] ndarray shape=(1, 17) dtype=float32 size=17 bytes=68 | min=-1.11215 max=3.74496 nan=0 inf=0
    [[-0.011641 -0.009544  0.030563 ... -0.179266 -0.03675   3.744961]]
[2.feature_contribs] ndarray shape=(1, 16) dtype=float32 size=16 bytes=64 | min=-1.11215 max=2.31204 nan=0 inf=0
    [[-0.011641 -0.009544  0.030563 ...  2.312041 -0.179266 -0.03675 ]]
[2.bias] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=3.74496 max=3.74496 nan=0 inf=0
    [3.744961]
[ok] bias + sum of contributions = margin: shape=(1,), max |diff| = 1.431e-06
[2.explanation.values] ndarray shape=(1, 16) dtype=float32 size=16 bytes=64 | min=-2.31204 max=1.11215 nan=0 inf=0
    [[ 0.011641  0.009544 -0.030563 ... -2.312041  0.179266  0.03675 ]]
[2.explanation.base_values] ndarray shape=(1,) dtype=float64 size=1 bytes=8 | min=-3.74496 max=-3.74496 nan=0 inf=0
    [-3.744961]
[ok] explain values = -contributions: shape=(1, 16), max |diff| = 0.000e+00
[ok] explain base value = -bias: shape=(1,), max |diff|

,value,contribution to margin,contribution to risk
avg_wind_ms,6.28,2.31204,-2.31204
mean_temp_c,0.3,-1.11215,1.11215
hard_landings_per_100h,5.02,-0.179266,0.179266
firmware_major,3,-0.152244,0.152244
initial_battery_cycles,24,0.0955785,-0.0955785
avg_humidity_pct,48.7,0.0869008,-0.0869008
avg_flight_min,23.5,0.0740775,-0.0740775
operator_experience_yrs,1.44,-0.0716829,0.0716829
maintenance_interval_h,200,-0.069208,0.069208
daily_utilization_h,0.75,-0.0678706,0.0678706


### 2.6 The prediction does not depend on `duration_h` and `event`

Neither column is in the feature frame, so changing them must leave the margin and the survival curve untouched. This is what allows asking for $S(t)$ at any $t$, including times beyond the drone's current age.

In [14]:
shifted = drone.copy()
shifted["duration_h"] = 1.0
shifted["event"] = 1
same("margin does not depend on duration_h and event", MODEL._predict_margin(shifted), margin)
same(
    "survival does not depend on duration_h and event",
    MODEL.survival(shifted, TIMES),
    MODEL.survival(drone, TIMES),
)

[ok] margin does not depend on duration_h and event: shape=(1,), max |diff| = 0.000e+00
[ok] survival does not depend on duration_h and event: shape=(1, 3), max |diff| = 0.000e+00


## 3. Risk

`risk` is the ranking score: a higher value means an earlier expected failure. For Cox it is the margin itself, the log relative hazard. For AFT the sign is flipped, because a large $\log T$ means a late failure:

$$\mathrm{risk}(x) = -\hat f(x).$$

The two models use different scales (log hazard ratio against log hours), so `risk` values are comparable only within one model. It is the score the C-index ranks (section 10) and the one `risk_group` in the predictions table is cut from.

In [15]:
risk = MODEL.risk(drone)
same("risk = -margin", risk, -margin)
peek("3.risk", risk)
print(f"risk = -margin = {risk[0]:.6f}")

[ok] risk = -margin: shape=(1,), max |diff| = 0.000e+00
[3.risk] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=-4.57009 max=-4.57009 nan=0 inf=0
    [-4.570089]
risk = -margin = -4.570089


## 4. Cumulative hazard

`_cum_hazard_at` returns $H(t \mid x) = -\log S(t \mid x)$ before calibration.

For AFT the time enters through the standardised log-time

$$z = (\log t - \hat f(x)) / \sigma,$$

and `_aft_noise` returns $H(z)$ and its derivative $H'(z)$ for the noise distribution:

| noise | time to failure | $H(z)$ | $H'(z)$ |
| --- | --- | --- | --- |
| normal | log-normal | $-\log(1-\Phi(z))$ | $\varphi(z)/(1-\Phi(z))$ |
| logistic | log-logistic | $\log(1+e^{z})$ | $\mathrm{expit}(z)$ |
| extreme | Weibull | $e^{z}$ | $e^{z}$ |

Shapes: `np.log(TIMES)` is `(m,)` and `margin[:, None]` is `(n, 1)`, so $z$ broadcasts to `(n, m)`. `_cum_hazard_at` accepts the times as `(m,)`, shared by all rows, or as `(n, m)`, one set per row; it always returns `(n, m)`.

At $t = 0$, $\log t = -\infty$, so $z = -\infty$, $H = 0$ and $S = 1$: every curve starts at 1.

In [16]:
log_t = np.log(TIMES)
margin_col = margin[:, None]
z = (log_t - margin_col) / sigma
H_noise, dH_dz = _aft_noise(dist, z)
peek("4.log_t", log_t)
peek("4.margin_col", margin_col)
peek("4.z", z)
peek("4.H_noise", H_noise)
peek("4.dH_dz", dH_dz)

if dist == "logistic":
    H_closed, dH_closed = np.logaddexp(0.0, z), expit(z)
elif dist == "normal":
    log_sf = norm.logsf(z)
    H_closed = -log_sf
    dH_closed = np.exp(norm.logpdf(z) - log_sf)
else:
    H_closed = np.exp(z)
    dH_closed = H_closed
same("H(z) = closed form", H_noise, H_closed)
same("H'(z) = closed form", dH_dz, dH_closed)

H = MODEL._cum_hazard_at(margin, TIMES)
peek("4.H", H)
same("_cum_hazard_at", H, H_noise)

H_at_zero = MODEL._cum_hazard_at(margin, np.array([0.0]))
peek("4.H_at_zero", H_at_zero)
same("H(0) = 0", H_at_zero, np.zeros((len(drone), 1)))

pd.DataFrame({"t": TIMES, "log t": log_t, "z": z[0], "H(z)": H[0], "H'(z)": dH_dz[0]})

[4.log_t] ndarray shape=(3,) dtype=float64 size=3 bytes=24 | min=3.91202 max=5.68188 nan=0 inf=0
    [3.912023 5.265278 5.681878]
[4.margin_col] ndarray shape=(1, 1) dtype=float32 size=1 bytes=4 | min=4.57009 max=4.57009 nan=0 inf=0
    [[4.570089]]
[4.z] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=-0.891471 max=1.50612 nan=0 inf=0
    [[-0.891471  0.941761  1.506122]]
[4.H_noise] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.343627 max=1.70642 nan=0 inf=0
    [[0.343627 1.271022 1.706421]]
[4.dH_dz] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.290806 max=0.818486 nan=0 inf=0
    [[0.290806 0.719455 0.818486]]
[ok] H(z) = closed form: shape=(1, 3), max |diff| = 0.000e+00
[ok] H'(z) = closed form: shape=(1, 3), max |diff| = 0.000e+00
[4.H] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.343627 max=1.70642 nan=0 inf=0
    [[0.343627 1.271022 1.706421]]
[ok] _cum_hazard_at: shape=(1, 3), max |diff| = 0.000e+00
[4.H_at_zero] ndarray shape=

,t,log t,z,H(z),H'(z)
0,50,3.91202,-0.891471,0.343627,0.290806
1,193.5,5.26528,0.941761,1.27102,0.719455
2,293.5,5.68188,1.50612,1.70642,0.818486


## 5. Survival

`_survival_at` turns the cumulative hazard into the probability of still operating at $t$:

$$S(t \mid x) = \exp(-H).$$

If `calibration` is set, $H$ first goes through the gamma-frailty link. With variance $v \ge 10^{-8}$

$$\Lambda = \log(1 + v \, k \, H) / v,$$

and in the limit $v \to 0$ it is $\Lambda = kH$. The link corrects the curves for differences between drones that the features do not capture. `drone_survival.ipynb` resets `calibration` to `None` when the fitted variance is zero, which leaves the model as trained. `survival` does the same as `_survival_at` after computing the margin itself.

The cell prints which branch the loaded model takes. As a sanity check it also evaluates $S$ at $\exp(\hat f)$, where $z = 0$: for normal or logistic noise without calibration that is 0.5, so $\exp(\hat f)$ is the median time to failure.

In [17]:
if MODEL.calibration is None:
    H_used = H
    print("calibration is None: the cumulative hazard enters exp(-H) unchanged")
else:
    v = MODEL.calibration["frailty_variance"]
    k = MODEL.calibration["scale"]
    H_used = k * H if v < 1e-8 else np.log1p(v * k * H) / v
    H_linked, _ = _frailty_link(H, **MODEL.calibration)
    same("frailty link", H_used, H_linked)
    print(f"H replaced by log(1 + v*k*H) / v with v={v:.4g}, k={k:.4g}")

S_hand = np.exp(-H_used)
S_at = MODEL._survival_at(margin, TIMES)
S = MODEL.survival(drone, TIMES)
peek("5.H_used", H_used)
peek("5.S", S)
same("_survival_at", S_at, S_hand)
same("survival", S, S_hand)

t_at_z0 = np.exp(margin).astype(float)
S_at_z0 = MODEL.survival(drone, t_at_z0)
peek("5.S_at_z0", S_at_z0)
print(f"\nS at exp(margin) = {t_at_z0[0]:.2f} h: {S_at_z0[0, 0]:.6f}")
pd.DataFrame({"t": TIMES, "H": H_used[0], "S": S[0]})

calibration is None: the cumulative hazard enters exp(-H) unchanged
[5.H_used] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.343627 max=1.70642 nan=0 inf=0
    [[0.343627 1.271022 1.706421]]
[5.S] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.181514 max=0.709194 nan=0 inf=0
    [[0.709194 0.280545 0.181514]]
[ok] _survival_at: shape=(1, 3), max |diff| = 0.000e+00
[ok] survival: shape=(1, 3), max |diff| = 0.000e+00
[5.S_at_z0] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=0.5 max=0.5 nan=0 inf=0
    [[0.5]]

S at exp(margin) = 96.55 h: 0.500000


,t,H,S
0,50,0.343627,0.709194
1,193.5,1.27102,0.280545
2,293.5,1.70642,0.181514


## 6. Conditional survival

The age $a$ is the flight hours already survived. The probability of flying $u$ more hours with the features unchanged is

$$S(a+u \mid x) \,/\, S(a \mid x).$$

At $u = 0$ the ratio is 1. At $u = 100$ it is the probability of surviving the next 100 hours. Where $S(a) = 0$ the function returns NaN; an AFT model does not produce that at a finite age.

Shapes: `ages` holds one age per row and becomes a column `(n, 1)`; the look-ahead times `u` are shared by all rows and become a row `(1, m)`; their sum is the `(n, m)` matrix of absolute times at which $S$ is evaluated.

In [18]:
U = np.array([0.0, LOOKAHEAD_H])
ages = np.array([AGE])
ages_col = ages[:, None]
u_row = U[None, :]
future = ages_col + u_row
s_age = MODEL._survival_at(margin, ages_col)
s_ahead = MODEL._survival_at(margin, future)
ratio = s_ahead / s_age
cond = MODEL.conditional_survival(drone, ages, U)
peek("6.ages_col", ages_col)
peek("6.u_row", u_row)
peek("6.future", future)
peek("6.s_age", s_age)
peek("6.s_ahead", s_ahead)
peek("6.cond", cond)
same("conditional_survival", cond, ratio)

pd.DataFrame(
    {
        "u": U,
        "a + u": future[0],
        "S(a)": np.full(len(U), s_age[0, 0]),
        "S(a+u)": s_ahead[0],
        "S(a+u)/S(a)": cond[0],
    }
)

[6.ages_col] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=193.5 max=193.5 nan=0 inf=0
    [[193.5]]
[6.u_row] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0 max=100 nan=0 inf=0
    [[  0. 100.]]
[6.future] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=193.5 max=293.5 nan=0 inf=0
    [[193.5 293.5]]
[6.s_age] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=0.280545 max=0.280545 nan=0 inf=0
    [[0.280545]]
[6.s_ahead] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0.181514 max=0.280545 nan=0 inf=0
    [[0.280545 0.181514]]
[6.cond] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0.647006 max=1 nan=0 inf=0
    [[1.       0.647006]]
[ok] conditional_survival: shape=(1, 2), max |diff| = 0.000e+00


,u,a + u,S(a),S(a+u),S(a+u)/S(a)
0,0,193.5,0.280545,0.280545,1
1,100,293.5,0.280545,0.181514,0.647006


## 7. Hazard

The hazard is the failure rate per flight hour among drones still operating at $t$. For AFT it is the derivative of the noise hazard, moved from the $z$ scale to hours:

$$h(t \mid x) = H'(z) \,/\, (\sigma \, t).$$

If `calibration` is set, the result is multiplied by the slope of the link, $k / (1 + v k H)$. The cell prints which of the two cases applies.

An independent check follows from the definition: the hazard is the derivative of the cumulative hazard, $h(t) = \mathrm{d}[-\log S(t)]/\mathrm{d}t$. The cell differentiates `survival` numerically with a central difference and compares. This covers the link as well, because `survival` already includes it.

In [19]:
h_noise = dH_dz / (sigma * TIMES)
if MODEL.calibration is None:
    link_slope = np.ones_like(H)
    print("no calibration: the slope factor is 1")
else:
    _, link_slope = _frailty_link(H, **MODEL.calibration)
    print("hazard multiplied by the slope of the link")
h_hand = link_slope * h_noise
h = MODEL.hazard(drone, TIMES)
peek("7.h_noise", h_noise)
peek("7.link_slope", link_slope)
peek("7.h", h)
same("hazard", h, h_hand)

step_h = 1e-3
H_after = -np.log(MODEL.survival(drone, TIMES + step_h))
H_before = -np.log(MODEL.survival(drone, TIMES - step_h))
h_numeric = (H_after - H_before) / (2 * step_h)
peek("7.h_numeric", h_numeric)
same("hazard = d[-log S]/dt, central difference", h, h_numeric, rtol=1e-6, atol=0)

pd.DataFrame(
    {
        "t": TIMES,
        "H'(z)": dH_dz[0],
        "h, per hour": h[0],
        f"h, per {HAZARD_SCALE:g} h": HAZARD_SCALE * h[0],
    }
)

no calibration: the slope factor is 1
[7.h_noise] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00377782 max=0.00787901 nan=0 inf=0
    [[0.007879 0.005037 0.003778]]
[7.link_slope] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=1 max=1 nan=0 inf=0
    [[1. 1. 1.]]
[7.h] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00377782 max=0.00787901 nan=0 inf=0
    [[0.007879 0.005037 0.003778]]
[ok] hazard: shape=(1, 3), max |diff| = 0.000e+00
[7.h_numeric] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00377782 max=0.00787901 nan=0 inf=0
    [[0.007879 0.005037 0.003778]]
[ok] hazard = d[-log S]/dt, central difference: shape=(1, 3), max |diff| = 3.984e-13


,t,H'(z),"h, per hour","h, per 100 h"
0,50,0.290806,0.00787901,0.787901
1,193.5,0.719455,0.00503687,0.503687
2,293.5,0.818486,0.00377782,0.377782


## 8. The observation record `_surv`

`_surv` packs `event` and `duration_h` into a structured array of `(event, time)` records, the format the `sksurv` metrics expect. It is the **observation**, not a prediction.

On a data set the metrics compare predictions with these records. Brier score and AUC compare `survival` at a horizon with whether the drone had failed by then. The C-index checks that in a pair of drones the one that failed earlier has the higher `risk`. A single row is never subtracted from anything; section 10 runs the comparison on the whole test table.

`event = False` means the record is censored: the drone was still flying at `time`.

In [20]:
label = _surv(drone)
peek("8.label", label)
print(f"fields: {label.dtype.names}; {label.dtype.itemsize} bytes per record")
print(f"record: event={bool(label['event'][0])}, time={float(label['time'][0]):g} h")

s_now = float(MODEL.survival(drone, ages)[0, 0])
peek("8.s_now", s_now)
print(f"S({AGE:g}) = {s_now:.6f}: the curve at the time of this record")
print(f"risk = {risk[0]:.6f}: the score the C-index compares between records")

[8.label] ndarray shape=(1,) dtype=[('event', '?'), ('time', '<f8')] size=1 bytes=9
    [(False, 193.5)]
fields: ('event', 'time'); 9 bytes per record
record: event=False, time=193.5 h
[8.s_now] float = 0.2805448604250952
S(193.5) = 0.280545: the curve at the time of this record
risk = -4.570089: the score the C-index compares between records


## 9. The same row through the Cox model

### 9.1 Model, margin and risk

The Cox model is read from the file named in `meta.json`. Its margin is the log relative hazard, and `risk` equals it without a sign change. $\exp(\hat f)$ is the hazard ratio of this drone against the baseline.

In [21]:
COX = Model.load(MODELS_DIR / META["production_models"]["cox"])
peek("9.COX.params", COX.params, limit=len(COX.params))
peek("9.COX.calibration", COX.calibration)
peek("9.COX._breslow", COX._breslow)
print(
    f"\nkind={COX.kind}; boosting rounds stored: {COX.booster.num_boosted_rounds()}; "
    f"best_iteration: {COX.booster.best_iteration}"
)

cox_margin = COX._predict_margin(drone)
cox_risk = COX.risk(drone)
hazard_ratio = np.exp(cox_margin)
peek("9.cox_margin", cox_margin)
peek("9.hazard_ratio", hazard_ratio)
same("Cox risk = margin", cox_risk, cox_margin)
print(f"margin = log relative hazard = {cox_margin[0]:.6f}")
print(f"hazard ratio exp(margin) = {hazard_ratio[0]:.4f}")

[9.COX.params] dict size=6
    'max_depth': 3
    'eta': 0.10161123659069494
    'min_child_weight': 2.31735485138156
    'subsample': 0.7835346123945779
    'colsample_bytree': 0.9582043558653918
    'reg_lambda': 1.8777925383998129
[9.COX.calibration] dict size=2
    'frailty_variance': 0.2678234346812382
    'scale': 1.0453121212260807
[9.COX._breslow] tuple size=2
    [0] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344
    [1] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344

kind=cox; boosting rounds stored: 118; best_iteration: 117
[9.cox_margin] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=1.67805 max=1.67805 nan=0 inf=0
    [1.678049]
[9.hazard_ratio] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=5.3551 max=5.3551 nan=0 inf=0
    [5.355099]
[ok] Cox risk = margin: shape=(1,), max |diff| = 0.000e+00
margin = log relative hazard = 1.678049
hazard ratio exp(margin) = 5.3551


### 9.2 Breslow baseline and cumulative hazard

`COX._breslow` is a pair of arrays of length `k`: the distinct failure times of the rows the model was fitted on (train + validation here) and the cumulative baseline hazard $H_0$ at each of them. $H_0$ is a step function: at time $t$ it takes the value at the last training failure time not later than $t$, and 0 before the first one. Then

$$H(t \mid x) = \exp(\hat f(x)) \, H_0(t).$$

`np.searchsorted(ev_times, t, side="right")` counts the failure times $\le t$; subtracting 1 gives the index of the step in force.

In [22]:
ev_times, h0 = COX._breslow
peek("9.ev_times", ev_times)
peek("9.h0", h0)
print(
    f"failure times strictly increasing: {bool(np.all(np.diff(ev_times) > 0))}; "
    f"H0 non-decreasing: {bool(np.all(np.diff(h0) >= 0))}"
)

step_count = np.searchsorted(ev_times, TIMES, side="right")
step_index = np.maximum(step_count - 1, 0)
h0_at = np.where(step_count > 0, h0[step_index], 0.0)
step_time = np.where(step_count > 0, ev_times[step_index], np.nan)
cox_H_hand = np.exp(cox_margin)[:, None] * h0_at
cox_H = COX._cum_hazard_at(cox_margin, TIMES)
peek("9.step_count", step_count)
peek("9.step_index", step_index)
peek("9.h0_at", h0_at)
peek("9.cox_H", cox_H)
same("Cox _cum_hazard_at", cox_H, cox_H_hand)

pd.DataFrame(
    {
        "t": TIMES,
        "failure times <= t": step_count,
        "last failure time <= t": step_time,
        "H0(t)": h0_at,
        "H(t)": cox_H[0],
    }
)

[9.ev_times] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344 | min=0.03 max=2347.64 nan=0 inf=0
    [3.00000e-02 9.00000e-02 1.50000e-01 ... 2.09355e+03 2.23751e+03 2.34764e+03]
[9.h0] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344 | min=0.000236034 max=6.2296 nan=0 inf=0
    [2.360344e-04 4.722465e-04 5.941000e-04 ... 5.291338e+00 5.704019e+00 6.229601e+00]
failure times strictly increasing: True; H0 non-decreasing: True
[9.step_count] ndarray shape=(3,) dtype=int64 size=3 bytes=24 | min=411 max=1501
    [ 411 1104 1501]
[9.step_index] ndarray shape=(3,) dtype=int64 size=3 bytes=24 | min=410 max=1500
    [ 410 1103 1500]
[9.h0_at] ndarray shape=(3,) dtype=float64 size=3 bytes=24 | min=0.0692587 max=0.431829 nan=0 inf=0
    [0.069259 0.259815 0.431829]
[9.cox_H] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.370887 max=2.31249 nan=0 inf=0
    [[0.370887 1.391337 2.312485]]
[ok] Cox _cum_hazard_at: shape=(1, 3), max |diff| = 0.000e+00


,t,failure times <= t,last failure time <= t,H0(t),H(t)
0,50,411,49.89,0.0692587,0.370887
1,193.5,1104,193.5,0.259815,1.39134
2,293.5,1501,293.49,0.431829,2.31249


### 9.3 Frailty link and survival

As in section 5: if the Cox model carries a `calibration`, $H$ goes through $H \to \log(1 + v k H)/v$ before $\exp(-H)$. The slope of the link, $k/(1 + v k H)$, is kept for the hazard in the next step.

In [23]:
if COX.calibration is None:
    cox_H_used = cox_H
    cox_link_slope = np.ones_like(cox_H)
    print("calibration is None: H enters exp(-H) unchanged")
else:
    cox_v = COX.calibration["frailty_variance"]
    cox_k = COX.calibration["scale"]
    if cox_v < 1e-8:
        cox_H_used = cox_k * cox_H
        cox_slope_hand = np.full_like(cox_H, cox_k)
    else:
        cox_H_used = np.log1p(cox_v * cox_k * cox_H) / cox_v
        cox_slope_hand = cox_k / (1.0 + cox_v * cox_k * cox_H)
    cox_H_linked, cox_link_slope = _frailty_link(cox_H, **COX.calibration)
    same("Cox frailty link", cox_H_used, cox_H_linked)
    same("Cox slope of the link", cox_slope_hand, cox_link_slope)
    print(f"H replaced by log(1 + v*k*H) / v with v={cox_v:.4g}, k={cox_k:.4g}")

cox_S_hand = np.exp(-cox_H_used)
cox_S = COX.survival(drone, TIMES)
peek("9.cox_H_used", cox_H_used)
peek("9.cox_link_slope", cox_link_slope)
peek("9.cox_S", cox_S)
same("Cox _survival_at", COX._survival_at(cox_margin, TIMES), cox_S_hand)
same("Cox survival", cox_S, cox_S_hand)

pd.DataFrame(
    {
        "t": TIMES,
        "H": cox_H[0],
        "H after the link": cox_H_used[0],
        "slope of the link": cox_link_slope[0],
        "S": cox_S[0],
    }
)

[ok] Cox frailty link: shape=(1, 3), max |diff| = 2.220e-16
[ok] Cox slope of the link: shape=(1, 3), max |diff| = 1.110e-16
H replaced by log(1 + v*k*H) / v with v=0.2678, k=1.045
[9.cox_H_used] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.368858 max=1.86391 nan=0 inf=0
    [[0.368858 1.228259 1.863911]]
[9.cox_link_slope] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.634522 max=0.946984 nan=0 inf=0
    [[0.946984 0.752284 0.634522]]
[9.cox_S] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.155065 max=0.691523 nan=0 inf=0
    [[0.691523 0.292802 0.155065]]
[ok] Cox _survival_at: shape=(1, 3), max |diff| = 2.776e-17
[ok] Cox survival: shape=(1, 3), max |diff| = 2.776e-17


,t,H,H after the link,slope of the link,S
0,50,0.370887,0.368858,0.946984,0.691523
1,193.5,1.39134,1.22826,0.752284,0.292802
2,293.5,2.31249,1.86391,0.634522,0.155065


### 9.4 Kernel-smoothed hazard

$H_0$ is a step function, so its derivative is a set of spikes at the training failure times. `hazard` spreads each jump of $H_0$ with an Epanechnikov kernel of half-width `bandwidth` $b$, scales by $\exp(\hat f)$ and multiplies by the slope of the link:

$$h(t \mid x) = \frac{k}{1 + v k H(t \mid x)} \; e^{\hat f(x)} \sum_j K_b(t - t_j)\, \Delta H_0(t_j), \qquad K_b(d) = \frac{0.75}{b}\Big(1 - \frac{d^2}{b^2}\Big)\ \text{for}\ |d| < b.$$

Shapes: the jumps $\Delta H_0$ are `(k,)`. The kernel weights form an `(m, k)` matrix with one row per evaluation time and one column per training failure time; most entries are zero, because only failure times within one bandwidth of $t$ contribute. The matrix product gives the smoothed baseline hazard, `(m,)`.

The bandwidth is an argument of `hazard`, not part of the model. This section takes Silverman's rule on the training failure times. The dashboard takes the rule on the test table (section 11), so its Cox hazard is slightly different. The estimate is biased downwards within one bandwidth of 0 and of the last training failure time, where part of the kernel window is empty.

In [24]:
cox_bandwidth = hazard_bandwidth(ev_times, np.ones(len(ev_times), dtype=bool))
h0_jumps = np.diff(h0, prepend=0.0)
kernel = _epanechnikov(TIMES, ev_times, cox_bandwidth)
kernel_support = (kernel > 0).sum(axis=1)
baseline_hazard = kernel @ h0_jumps
cox_h_unlinked = np.exp(cox_margin)[:, None] * baseline_hazard[None, :]
cox_h_hand = cox_link_slope * cox_h_unlinked
cox_h = COX.hazard(drone, TIMES, bandwidth=cox_bandwidth)
peek("9.cox_bandwidth", cox_bandwidth)
peek("9.h0_jumps", h0_jumps)
peek("9.kernel", kernel)
peek("9.kernel_support", kernel_support)
peek("9.baseline_hazard", baseline_hazard)
peek("9.cox_h_unlinked", cox_h_unlinked)
peek("9.cox_h", cox_h)
same("Cox hazard", cox_h, cox_h_hand)

pd.DataFrame(
    {
        "t": TIMES,
        "failure times in the window": kernel_support,
        "baseline hazard": baseline_hazard,
        "slope of the link": cox_link_slope[0],
        "h, per hour": cox_h[0],
        f"h, per {HAZARD_SCALE:g} h": HAZARD_SCALE * cox_h[0],
    }
)

[9.cox_bandwidth] float = 49.371924407013864
[9.h0_jumps] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344 | min=0.000121854 max=0.525582 nan=0 inf=0
    [2.360344e-04 2.362121e-04 1.218535e-04 ... 2.597865e-01 4.126814e-01 5.255822e-01]
[9.kernel] ndarray shape=(3, 2543) dtype=float64 size=7629 bytes=61032 | min=0 max=0.0151908 nan=0 inf=0
    [[0. 0. 0. ... 0. 0. 0.]
     [0. 0. 0. ... 0. 0. 0.]
     [0. 0. 0. ... 0. 0. 0.]]
[9.kernel_support] ndarray shape=(3,) dtype=int64 size=3 bytes=24 | min=323 max=675
    [675 409 323]
[9.baseline_hazard] ndarray shape=(3,) dtype=float64 size=3 bytes=24 | min=0.00128666 max=0.00170974 nan=0 inf=0
    [0.001287 0.00146  0.00171 ]
[9.cox_h_unlinked] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00689021 max=0.00915583 nan=0 inf=0
    [[0.00689  0.007821 0.009156]]
[9.cox_h] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00580958 max=0.00652492 nan=0 inf=0
    [[0.006525 0.005884 0.00581 ]]
[ok] Cox hazard: shape=

,t,failure times in the window,baseline hazard,slope of the link,"h, per hour","h, per 100 h"
0,50,675,0.00128666,0.946984,0.00652492,0.652492
1,193.5,409,0.00146045,0.752284,0.00588351,0.588351
2,293.5,323,0.00170974,0.634522,0.00580958,0.580958


### 9.5 Conditional survival and the end of the baseline

The ratio $S(a+u)/S(a)$ is computed as in section 6. One trap is specific to Cox: $H_0$ is flat after the last training failure time, so beyond it the model predicts no further failures and the conditional curve is too optimistic. `conditional_survival` logs a warning when $a + u$ passes that time. The call with $u = 0$ and $u = 100$ stays inside the baseline; a separate call one hour past the last training failure triggers the warning.

The last table puts the two models side by side at the probe times.

In [25]:
cox_s_age = COX._survival_at(cox_margin, ages_col)
cox_s_ahead = COX._survival_at(cox_margin, future)
cox_cond = COX.conditional_survival(drone, ages, U)
peek("9.cox_s_age", cox_s_age)
peek("9.cox_s_ahead", cox_s_ahead)
peek("9.cox_cond", cox_cond)
same("Cox conditional_survival", cox_cond, cox_s_ahead / cox_s_age)

last_train_failure = float(ev_times[-1])
past_u = np.array([last_train_failure - AGE + 1.0])
print(
    f"\nlast training failure at {last_train_failure:.1f} h; "
    f"a call with u={past_u[0]:.1f} h goes past the baseline:"
)
cox_cond_past = COX.conditional_survival(drone, ages, past_u)
peek("9.cox_cond_past", cox_cond_past)

display(
    pd.DataFrame(
        {
            "u": U,
            "S(a)": np.full(len(U), cox_s_age[0, 0]),
            "S(a+u)": cox_s_ahead[0],
            "S(a+u)/S(a)": cox_cond[0],
        }
    )
)
pd.DataFrame(
    {
        "t": TIMES,
        "S AFT": S[0],
        "S Cox": cox_S[0],
        f"h AFT, per {HAZARD_SCALE:g} h": HAZARD_SCALE * h[0],
        f"h Cox, per {HAZARD_SCALE:g} h": HAZARD_SCALE * cox_h[0],
    }
)

[9.cox_s_age] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=0.292802 max=0.292802 nan=0 inf=0
    [[0.292802]]
[9.cox_s_ahead] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0.155065 max=0.292802 nan=0 inf=0
    [[0.292802 0.155065]]
[9.cox_cond] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0.52959 max=1 nan=0 inf=0
    [[1.      0.52959]]
[ok] Cox conditional_survival: shape=(1, 2), max |diff| = 0.000e+00

last training failure at 2347.6 h; a call with u=2155.1 h goes past the baseline:
Cox baseline hazard is flat after 2348 h; conditional survival beyond it is optimistic


[9.cox_cond_past] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=0.00055653 max=0.00055653 nan=0 inf=0
    [[0.000557]]


,u,S(a),S(a+u),S(a+u)/S(a)
0,0,0.292802,0.292802,1
1,100,0.292802,0.155065,0.52959


,t,S AFT,S Cox,"h AFT, per 100 h","h Cox, per 100 h"
0,50,0.709194,0.691523,0.787901,0.652492
1,193.5,0.280545,0.292802,0.503687,0.588351
2,293.5,0.181514,0.155065,0.377782,0.580958


## 10. The whole test table

Everything above used `n = 1`. The dashboard and the metrics work on `artifacts/predictions.parquet`: the test split with the predictions added by `drone_survival.ipynb` (`risk_cox`, `risk_aft` of the evaluation models; `forecast_risk_cox`, `forecast_risk_aft` of the production models; `risk_group`, the remaining-life columns and `priority` of the production model named by `forecast_name`).

### 10.1 Stored predictions against the models on disk

The same methods now return one row per drone: `risk` is `(n,)` and `survival` is `(n, m)`. Two checks:

- the stored `forecast_risk_*` columns equal `Model.risk` of the production model files and the `risk_*` columns that of the evaluation model files, so the table and the models come from the same run;
- if the traced drone is in the test table, its row of the batch result equals the single-row prediction of the sections above, so batching changes nothing.

The evaluation times are `eval_times` from `meta.json`, the grid the metrics were computed on.

In [26]:
pred = pd.read_parquet(ARTIFACTS_DIR / "predictions.parquet")
MODELS = {"cox": COX, "aft": MODEL}
EVAL_MODELS = {kind: Model.load(MODELS_DIR / name) for kind, name in META["models"].items()}
EVAL_TIMES = np.array(META["eval_times"]["test"])
peek("10.pred", pred)
peek("10.EVAL_TIMES", EVAL_TIMES)

drone_positions = np.flatnonzero(pred["drone_id"].to_numpy() == DRONE_ID)
IN_TEST = len(drone_positions) == 1
DRONE_POS = int(drone_positions[0]) if IN_TEST else None
print(f"{DRONE_ID} in the test table: {IN_TEST}" + (f", row {DRONE_POS}" if IN_TEST else ""))

BATCH_RISK = {kind: model.risk(pred) for kind, model in MODELS.items()}
BATCH_SURV = {kind: model.survival(pred, EVAL_TIMES) for kind, model in MODELS.items()}
for kind, model in MODELS.items():
    peek(f"10.BATCH_RISK.{kind}", BATCH_RISK[kind])
    peek(f"10.BATCH_SURV.{kind}", BATCH_SURV[kind])
    same(
        f"{kind}: stored forecast_risk_{kind} = production Model.risk",
        pred[f"forecast_risk_{kind}"],
        BATCH_RISK[kind],
    )
    same(
        f"{kind}: stored risk_{kind} = evaluation Model.risk",
        pred[f"risk_{kind}"],
        EVAL_MODELS[kind].risk(pred),
    )
    if IN_TEST:
        row = slice(DRONE_POS, DRONE_POS + 1)
        same(f"{kind}: batch row {DRONE_POS} = single-row risk", BATCH_RISK[kind][row], model.risk(drone))
        same(
            f"{kind}: batch row {DRONE_POS} = single-row survival",
            BATCH_SURV[kind][row],
            model.survival(drone, EVAL_TIMES),
        )

pred.loc[pred["drone_id"] == DRONE_ID].T

[10.pred] DataFrame shape=(1001, 33) dtype=16 x float64, 6 x str, 4 x int64, 4 x float32, 2 x bool, 1 x datetime64[us] size=33033 bytes=281024
    columns: drone_id, model, environment, commission_date, is_refurbished, initial_battery_cycles,
    firmware_major, maintenance_interval_h, operator_experience_yrs, daily_utilization_h,
    avg_flight_min, avg_payload_kg, payload_ratio, mean_temp_c, avg_humidity_pct, avg_wind_ms,
    hard_landings_per_100h, mean_vibration_g, duration_h, event, failure_mode, in_service, risk_cox,
    risk_aft, forecast_risk_cox, forecast_risk_aft, risk_group, age_h, rul_q10, rul_median, rmrl,
    p_fail_window, priority
[10.EVAL_TIMES] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=21.4565 max=385.48 nan=0 inf=0
    [ 21.4565    28.885551  36.314602 ... 370.621898 378.050949 385.48    ]
DR-003057 in the test table: True, row 613


[10.BATCH_RISK.cox] ndarray shape=(1001,) dtype=float32 size=1001 bytes=4004 | min=-1.38264 max=4.96308 nan=0 inf=0
    [-0.759229 -0.459004 -0.074707 ...  0.196598 -0.900121  0.878095]
[10.BATCH_SURV.cox] ndarray shape=(1001, 50) dtype=float64 size=50050 bytes=400400 | min=6.43041e-06 max=0.992524 nan=0 inf=0
    [[0.986112 0.981603 0.977164 ... 0.767834 0.763061 0.757715]
     [0.981307 0.975263 0.969325 ... 0.702971 0.697216 0.690787]
     [0.972701 0.963939 0.955362 ... 0.602422 0.595487 0.587776]
     ...
     [0.964386 0.953037 0.941967 ... 0.521182 0.513627 0.505259]
     [0.987923 0.983996 0.980127 ... 0.794127 0.789798 0.784945]
     [0.931135 0.90983  0.889341 ... 0.302903 0.295446 0.287301]]
[ok] cox: stored forecast_risk_cox = production Model.risk: shape=(1001,), max |diff| = 0.000e+00
[ok] cox: stored risk_cox = evaluation Model.risk: shape=(1001,), max |diff| = 0.000e+00
[ok] cox: batch row 613 = single-row risk: shape=(1,), max |diff| = 0.000e+00
[ok] cox: batch row 613

,613
drone_id,DR-003057
model,quad_light
environment,urban
commission_date,2025-04-17 00:00:00
is_refurbished,False
initial_battery_cycles,24
firmware_major,3
maintenance_interval_h,200
operator_experience_yrs,1.44
daily_utilization_h,0.75


### 10.2 What the metrics receive

`utils.evaluate` passes three arrays to `sksurv`:

- `_surv(data)`: the observed `(event, time)` records, shape `(n,)`;
- `model.survival(data, times)`: predicted $S(t)$ on the evaluation grid, shape `(n, m)`; the Brier score uses it directly and the AUC as $1 - S$;
- `model.risk(data)`: the ranking score, shape `(n,)`; both C-indices use it.

The censoring distribution for the IPCW weights is estimated on the evaluated rows themselves, as in `drone_survival.ipynb`. The reported metrics describe the **evaluation** models, so the cell recomputes the test metrics with them and compares the result with `artifacts/metrics.parquet`. If they agree, the models, the predictions table and the reported metrics are consistent with each other. `brier` and `auc` are per-time arrays of shape `(m,)`, the curves of the model-quality figures.

In [27]:
y_test = _surv(pred)
peek("10.y_test", y_test)
print(f"failures: {int(y_test['event'].sum())} of {len(y_test)} records")

stored_metrics = pd.read_parquet(ARTIFACTS_DIR / "metrics.parquet")
peek("10.stored_metrics", stored_metrics)

EVALUATION = {
    kind: evaluate(model, pred, pred, EVAL_TIMES) for kind, model in EVAL_MODELS.items()
}
for kind, result in EVALUATION.items():
    peek(f"10.EVALUATION.{kind}.brier", result["brier"])
    peek(f"10.EVALUATION.{kind}.auc", result["auc"])
    on_test = (stored_metrics["model"] == kind) & (stored_metrics["split"] == "test")
    stored = stored_metrics[on_test].iloc[0]
    for metric in SCALAR_METRICS:
        same(
            f"{kind} {metric} = metrics.parquet",
            np.array([result[metric]]),
            np.array([stored[metric]]),
            rtol=1e-9,
        )

pd.DataFrame(
    {
        kind: {metric: result[metric] for metric in SCALAR_METRICS}
        for kind, result in EVALUATION.items()
    }
).T

[10.y_test] ndarray shape=(1001,) dtype=[('event', '?'), ('time', '<f8')] size=1001 bytes=9009
    [( True,  63.67) ( True, 159.67) (False, 312.4 ) ... ( True,  66.78) (False, 227.58)
     ( True,   1.03)]
failures: 448 of 1001 records
[10.stored_metrics] DataFrame shape=(4, 6) dtype=4 x float64, 2 x str size=24 bytes=350
    columns: model, split, c_index, c_index_ipcw, mean_auc, ibs


[10.EVALUATION.cox.brier] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=0.0484388 max=0.221214 nan=0 inf=0
    [0.048439 0.057875 0.068015 ... 0.219005 0.218536 0.220104]
[10.EVALUATION.cox.auc] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=0.67903 max=0.713779 nan=0 inf=0
    [0.70626  0.712709 0.687843 ... 0.710284 0.713779 0.709326]
[ok] cox c_index = metrics.parquet: shape=(1,), max |diff| = 0.000e+00
[ok] cox c_index_ipcw = metrics.parquet: shape=(1,), max |diff| = 0.000e+00
[ok] cox mean_auc = metrics.parquet: shape=(1,), max |diff| = 0.000e+00
[ok] cox ibs = metrics.parquet: shape=(1,), max |diff| = 0.000e+00
[10.EVALUATION.aft.brier] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=0.0498959 max=0.227226 nan=0 inf=0
    [0.049896 0.059202 0.069606 ... 0.224129 0.223605 0.225016]
[10.EVALUATION.aft.auc] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=0.674577 max=0.699807 nan=0 inf=0
    [0.689536 0.699748 0.674577 ... 0.689008 0.691548

,c_index,c_index_ipcw,mean_auc,ibs
cox,0.671746,0.664759,0.699508,0.170899
aft,0.665767,0.658315,0.687146,0.175398


## 11. Dashboard settings and grids

### 11.1 Settings and the kernel bandwidth

The dashboard does not draw curves at the three probe times. `Settings.from_data` derives everything a figure needs from `meta.json` and the test predictions: the remaining-life `horizon` and `window`, the priority thresholds, three time grids and the kernel bandwidth.

`hazard_bandwidth` is Silverman's rule on the failure times of the test table:

$$b = 0.9 \cdot \min(\mathrm{sd},\ \mathrm{IQR}/1.34) \cdot n^{-1/5},$$

where $n$ is the number of failures. The cell recomputes it by hand.

In [28]:
DATA = DashboardData(
    meta=META, tables={"predictions": pred}, models=EVAL_MODELS, production=MODELS, card=CARD
)
SETTINGS = Settings.from_data(DATA)
FAILED = EVENT == 1
settings_fields = {
    field.name: getattr(SETTINGS, field.name) for field in dataclasses.fields(SETTINGS)
}
peek("11.SETTINGS", settings_fields, limit=len(settings_fields))

event_durations = pred.loc[pred["event"] == 1, "duration_h"].to_numpy(dtype=float)
q75, q25 = np.quantile(event_durations, [0.75, 0.25])
iqr = q75 - q25
sd = event_durations.std(ddof=1)
spread = min(sd, iqr / 1.34) if iqr > 0 else sd
bandwidth_hand = 0.9 * spread * len(event_durations) ** -0.2
peek("11.event_durations", event_durations)
same(
    "hazard_bandwidth = Silverman's rule",
    np.array([SETTINGS.hazard_bandwidth]),
    np.array([bandwidth_hand]),
)
print(
    f"sd={sd:.2f} h, IQR/1.34={iqr / 1.34:.2f} h, n={len(event_durations)} failures "
    f"-> bandwidth {bandwidth_hand:.2f} h"
)
print(
    f"age {AGE:g} h; window {SETTINGS.window:g} h; remaining-life horizon {SETTINGS.horizon:g} h; "
    f"observed failure: {FAILED}"
)

[11.SETTINGS] dict size=12
    'best': 'aft'
    'forecast': 'cox'
    'features': list size=16
    'cat_features': ['model', 'environment', 'is_refurbished', 'firmware_major']
    'horizon': 1111.0
    'window': 100.0
    'p_high': 0.5
    'p_medium': 0.2
    'surv_grid': ndarray shape=(300,) dtype=float64 size=300 bytes=2400
    'rul_grid': ndarray shape=(300,) dtype=float64 size=300 bytes=2400
    'hazard_bandwidth': 42.439288934251536
    'hazard_grid': ndarray shape=(200,) dtype=float64 size=200 bytes=1600
[11.event_durations] ndarray shape=(448,) dtype=float64 size=448 bytes=3584 | min=0.55 max=1156.68 nan=0 inf=0
    [ 63.67 159.67 130.12 ...  41.49  66.78   1.03]
[ok] hazard_bandwidth = Silverman's rule: shape=(1,), max |diff| = 0.000e+00
sd=159.88 h, IQR/1.34=164.95 h, n=448 failures -> bandwidth 42.44 h
age 193.5 h; window 100 h; remaining-life horizon 1111 h; observed failure: False


### 11.2 Grids

- `surv_grid`, survival curves: 300 points from 0 to the largest `duration_h` in the test table.
- `rul_grid`, conditional curves: 300 points from 0 to `rul_horizon` from `meta.json`. Its unit is additional flight hours, not total hours (section 14).
- `hazard_grid`, fleet hazard: 200 points from one bandwidth to the 90th percentile of `duration_h`. The smoothed estimate is biased near 0 and rests on too few drones in the tail, so the grid avoids both.
- `drone_hazard_grid(settings, age)`, hazard of one drone: 300 points with the same left edge; the right edge moves out to `age + window` when that is beyond the 90th percentile. The curve is not extended further, because out there the Cox baseline rests on few training failures.

The age usually falls between two grid nodes. The vertical line of a figure is drawn at the exact `AGE`, while the curve is a polyline through the nodes. The table gives, for each grid in total hours, the node nearest to the age and its offset.

In [29]:
same("surv_grid", SETTINGS.surv_grid, np.linspace(0.0, pred["duration_h"].max(), 300))
same("rul_grid", SETTINGS.rul_grid, np.linspace(0.0, META["rul_horizon"], 300))
same(
    "hazard_grid",
    SETTINGS.hazard_grid,
    np.linspace(SETTINGS.hazard_bandwidth, pred["duration_h"].quantile(0.9), 200),
)
hazard_end = max(float(SETTINGS.hazard_grid[-1]), AGE + SETTINGS.window)
DRONE_HAZARD_GRID = drone_hazard_grid(SETTINGS, AGE)
same(
    "drone_hazard_grid",
    DRONE_HAZARD_GRID,
    np.linspace(SETTINGS.hazard_bandwidth, hazard_end, 300),
)
peek("11.surv_grid", SETTINGS.surv_grid)
peek("11.rul_grid", SETTINGS.rul_grid)
peek("11.hazard_grid", SETTINGS.hazard_grid)
peek("11.DRONE_HAZARD_GRID", DRONE_HAZARD_GRID)


def nearest(grid: np.ndarray) -> tuple[int, float]:
    """Index and value of the node of ``grid`` closest to ``AGE``."""
    i = int(np.argmin(np.abs(grid - AGE)))
    return i, float(grid[i])


grid_rows = []
for name, grid in (
    ("surv_grid", SETTINGS.surv_grid),
    ("hazard_grid", SETTINGS.hazard_grid),
    ("drone_hazard_grid", DRONE_HAZARD_GRID),
):
    _, node = nearest(grid)
    grid_rows.append(
        {
            "grid": name,
            "points": len(grid),
            "from": float(grid[0]),
            "to": float(grid[-1]),
            "step": float(grid[1] - grid[0]),
            "node nearest to the age": node,
            "offset, h": node - AGE,
        }
    )
print(
    f"\nright edge of the drone hazard grid = max(90th percentile "
    f"{SETTINGS.hazard_grid[-1]:.1f}, age + window {AGE + SETTINGS.window:.1f}) = {hazard_end:.1f} h"
)
pd.DataFrame(grid_rows)

[ok] surv_grid: shape=(300,), max |diff| = 0.000e+00
[ok] rul_grid: shape=(300,), max |diff| = 0.000e+00
[ok] hazard_grid: shape=(200,), max |diff| = 0.000e+00
[ok] drone_hazard_grid: shape=(300,), max |diff| = 0.000e+00
[11.surv_grid] ndarray shape=(300,) dtype=float64 size=300 bytes=2400 | min=0 max=1236.24 nan=0 inf=0
    [   0.          4.134582    8.269164 ... 1227.970836 1232.105418 1236.24    ]
[11.rul_grid] ndarray shape=(300,) dtype=float64 size=300 bytes=2400 | min=0 max=1111 nan=0 inf=0
    [   0.          3.715719    7.431438 ... 1103.568562 1107.284281 1111.      ]
[11.hazard_grid] ndarray shape=(200,) dtype=float64 size=200 bytes=1600 | min=42.4393 max=495.71 nan=0 inf=0
    [ 42.439289  44.717031  46.994773 ... 491.154515 493.432258 495.71    ]
[11.DRONE_HAZARD_GRID] ndarray shape=(300,) dtype=float64 size=300 bytes=2400 | min=42.4393 max=495.71 nan=0 inf=0
    [ 42.439289  43.955244  45.4712   ... 492.678089 494.194044 495.71    ]

right edge of the drone hazard grid = 

,grid,points,from,to,step,node nearest to the age,"offset, h"
0,surv_grid,300,0,1236.24,4.13458,194.325,0.825351
1,hazard_grid,200,42.4393,495.71,2.27774,192.77,-0.729721
2,drone_hazard_grid,300,42.4393,495.71,1.51596,194.035,0.534844


## 12. Survival curve of one drone

`figures.drone_survival` adds one line per model with `x = surv_grid` and `y = model.survival(row, surv_grid)[0]`: `survival` returns `(1, 300)` for the single row, and `[0]` takes the `(300,)` curve. The y axis is $S(t)$ with a fixed range.

The vertical line marks `duration_h`: green and labelled "flown" for `event = 0`, red and labelled "failed at" once the failure has been observed. It is a layout shape, not a trace. Its x is a data value; its y runs from 0 to 1 in axis fractions (`y domain`), so it always spans the full height.

The drone card of the dashboard makes the same call: `drone_survival(FORECAST_MODELS, rows, SETTINGS.surv_grid, row["duration_h"], failed)`, where `FORECAST_MODELS` are the production models, `MODELS` here.

The cell builds the figure, lists its traces and marks, recomputes each curve and compares it with what the trace holds.

In [30]:
surv_fig = drone_survival(MODELS, drone, SETTINGS.surv_grid, AGE, FAILED)
describe_figure("12.surv_fig", surv_fig)

SURV_CURVES = {kind: model.survival(drone, SETTINGS.surv_grid) for kind, model in MODELS.items()}
for kind, curve in SURV_CURVES.items():
    peek(f"12.SURV_CURVES.{kind}", curve)
    drawn = trace_xy(surv_fig, MODEL_LABELS[kind])
    same(f"{kind}: trace x = surv_grid", drawn["x"], SETTINGS.surv_grid)
    same(f"{kind}: trace y = survival(surv_grid)[0]", drawn["y"], curve[0])

surv_vline = surv_fig.layout.shapes[0]
same(
    "vertical line at AGE",
    np.array([surv_vline.x0, surv_vline.x1], dtype=float),
    np.array([AGE, AGE]),
)
print("label of the vertical line:", surv_fig.layout.annotations[0].text)

[12.surv_fig] Figure: 2 traces, 1 shapes, 1 annotations
    title:  'Probability of operating without failure'
    x axis: 'Flight hours', range=None, rangemode=None
    y axis: 'S(t)', range=(0, 1.02), rangemode=None
    height: 400


,trace,name,type,mode,x shape,y shape,x min,x max,y min,y max,NaN in y,line colour,line width,fill,in legend
0,0,Cox,scatter,lines,"(300,)","(300,)",0,1236.24,0.00289175,1,0,#1f77b4,None,None,True
1,1,AFT,scatter,lines,"(300,)","(300,)",0,1236.24,0.0306471,1,0,#ff7f0e,None,None,True


,kind,xref,x0,x1,yref,y0,y1,colour,dash,text
0,shape/line,x,193.5,193.5,y domain,0,1,green,dash,NaN
1,annotation,x,193.5,193.5,y domain,1,1,NaN,NaN,flown 194 h


[12.SURV_CURVES.cox] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.00289175 max=1 nan=0 inf=0
    [[1.       0.968987 0.940087 ... 0.002999 0.002892 0.002892]]
[ok] cox: trace x = surv_grid: shape=(300,), max |diff| = 0.000e+00
[ok] cox: trace y = survival(surv_grid)[0]: shape=(300,), max |diff| = 0.000e+00
[12.SURV_CURVES.aft] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.0306471 max=1 nan=0 inf=0
    [[1.       0.986187 0.965417 ... 0.030918 0.030782 0.030647]]
[ok] aft: trace x = surv_grid: shape=(300,), max |diff| = 0.000e+00
[ok] aft: trace y = survival(surv_grid)[0]: shape=(300,), max |diff| = 0.000e+00
[ok] vertical line at AGE: shape=(2,), max |diff| = 0.000e+00
label of the vertical line: flown 194 h


$S$ at a grid node is a vertex of the polyline. $S(\mathrm{AGE})$ from section 8 lies on the vertical line and coincides with a vertex only if the age itself is a node of `surv_grid`. The cell shows the figure, the plotted values at the nodes around the age, and the exact values at the probe times.

In [31]:
surv_plot_table = pd.DataFrame(
    {"t": SETTINGS.surv_grid, **{f"S {kind}": curve[0] for kind, curve in SURV_CURVES.items()}}
)
peek("12.surv_plot_table", surv_plot_table)

surv_node_i, surv_node_t = nearest(SETTINGS.surv_grid)
print(
    f"node of surv_grid nearest to the age: {surv_node_t:.4f} h, "
    f"S(AFT) = {SURV_CURVES['aft'][0, surv_node_i]:.6f}; exact S({AGE:g}) = {s_now:.6f}"
)
display(surv_fig)
print("plotted values at the nodes around the age:")
display(surv_plot_table.iloc[max(surv_node_i - 2, 0) : surv_node_i + 3])
print("exact values at the probe times:")
pd.DataFrame(
    {"t": TIMES, **{f"S {kind}": model.survival(drone, TIMES)[0] for kind, model in MODELS.items()}}
)

[12.surv_plot_table] DataFrame shape=(300, 3) dtype=3 x float64 size=900 bytes=7332
    columns: t, S cox, S aft
node of surv_grid nearest to the age: 194.3254 h, S(AFT) = 0.279383; exact S(193.5) = 0.280545


plotted values at the nodes around the age:


,t,S cox,S aft
45,186.056,0.304167,0.291395
46,190.191,0.2974,0.285286
47,194.325,0.291146,0.279383
48,198.46,0.282562,0.273677
49,202.595,0.274938,0.26816


exact values at the probe times:


,t,S cox,S aft
0,50,0.691523,0.709194
1,193.5,0.292802,0.280545
2,293.5,0.155065,0.181514


## 13. Hazard curve of one drone

`figures.drone_hazard` does not draw $h(t)$ itself but `HAZARD_SCALE * model.hazard(row, grid, bandwidth)[0]` with `HAZARD_SCALE = 100`. The axis label "Failures per 100 flight hours" is a failure rate per 100 flight hours.

The x values are `drone_hazard_grid(SETTINGS, age)`. The Cox line uses `SETTINGS.hazard_bandwidth`, so it differs from the Cox hazard of section 9.4, which was computed with another bandwidth. The vertical line is the same kind of shape as in section 12.

The left edge of the grid is one bandwidth, not zero. A probe time to the left of it has a hazard value in the rate table further down but no point on the line.

In [32]:
hazard_fig = drone_hazard(
    MODELS, drone, DRONE_HAZARD_GRID, SETTINGS.hazard_bandwidth, AGE, FAILED
)
describe_figure("13.hazard_fig", hazard_fig)

HAZARD_CURVES = {
    kind: model.hazard(drone, DRONE_HAZARD_GRID, SETTINGS.hazard_bandwidth)
    for kind, model in MODELS.items()
}
for kind, curve in HAZARD_CURVES.items():
    peek(f"13.HAZARD_CURVES.{kind}", curve)
    drawn = trace_xy(hazard_fig, MODEL_LABELS[kind])
    same(f"{kind}: trace x = drone_hazard_grid", drawn["x"], DRONE_HAZARD_GRID)
    same(f"{kind}: trace y = {HAZARD_SCALE:g} * hazard", drawn["y"], HAZARD_SCALE * curve[0])

hazard_vline = hazard_fig.layout.shapes[0]
same(
    "vertical line at AGE",
    np.array([hazard_vline.x0, hazard_vline.x1], dtype=float),
    np.array([AGE, AGE]),
)
display(hazard_fig)

[13.hazard_fig] Figure: 2 traces, 1 shapes, 1 annotations
    title:  'Failure rate of this drone by flight hours'
    x axis: 'Flight hours', range=None, rangemode=None
    y axis: 'Failures per 100 flight hours', range=None, rangemode=tozero
    height: 400


,trace,name,type,mode,x shape,y shape,x min,x max,y min,y max,NaN in y,line colour,line width,fill,in legend
0,0,Cox,scatter,lines,"(300,)","(300,)",42.4393,495.71,0.448933,0.668797,0,#1f77b4,None,None,True
1,1,AFT,scatter,lines,"(300,)","(300,)",42.4393,495.71,0.246415,0.789521,0,#ff7f0e,None,None,True


,kind,xref,x0,x1,yref,y0,y1,colour,dash,text
0,shape/line,x,193.5,193.5,y domain,0,1,green,dash,NaN
1,annotation,x,193.5,193.5,y domain,1,1,NaN,NaN,flown 194 h


[13.HAZARD_CURVES.cox] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.00448933 max=0.00668797 nan=0 inf=0
    [[0.006594 0.006558 0.006522 ... 0.004489 0.004505 0.004517]]
[ok] cox: trace x = drone_hazard_grid: shape=(300,), max |diff| = 0.000e+00
[ok] cox: trace y = 100 * hazard: shape=(300,), max |diff| = 0.000e+00
[13.HAZARD_CURVES.aft] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.00246415 max=0.00789521 nan=0 inf=0
    [[0.007891 0.007895 0.007895 ... 0.002477 0.002471 0.002464]]
[ok] aft: trace x = drone_hazard_grid: shape=(300,), max |diff| = 0.000e+00
[ok] aft: trace y = 100 * hazard: shape=(300,), max |diff| = 0.000e+00
[ok] vertical line at AGE: shape=(2,), max |diff| = 0.000e+00


**Rate now against probability over the window.** The height of the line at `AGE` is a rate, not the probability of failing soon. The probability of failure within the next `window` hours is the quantity of section 6:

$$1 - S(a + w \mid x) \,/\, S(a \mid x).$$

If the rate stayed constant it would be $1 - e^{-h(a)\,w}$. The curve rises or falls over the window, so the height at `AGE` alone does not give that probability. The cell prints both numbers for each model, and marks which of the probe times lie inside the plotted range.

In [33]:
window = float(SETTINGS.window)
window_arr = np.array([window])
in_range = (TIMES >= DRONE_HAZARD_GRID[0]) & (TIMES <= DRONE_HAZARD_GRID[-1])
rate_columns = {"t": TIMES, "inside the plotted range": in_range}
window_rows = []
for kind, model in MODELS.items():
    rate = model.hazard(drone, TIMES, SETTINGS.hazard_bandwidth)[0]
    rate_columns[f"h {kind}, per hour"] = rate
    rate_columns[f"h {kind}, per {HAZARD_SCALE:g} h"] = HAZARD_SCALE * rate
    s_at_age = model.survival(drone, ages)[0, 0]
    s_at_window = model.survival(drone, ages + window)[0, 0]
    p_fail = 1.0 - s_at_window / s_at_age
    p_fail_cond = 1.0 - model.conditional_survival(drone, ages, window_arr)[0, 0]
    same(f"{kind}: 1 - S(a+w)/S(a)", np.array([p_fail]), np.array([p_fail_cond]))
    rate_now = float(model.hazard(drone, ages, SETTINGS.hazard_bandwidth)[0, 0])
    window_rows.append(
        {
            "model": kind,
            f"rate at the age, per {HAZARD_SCALE:g} h": HAZARD_SCALE * rate_now,
            f"P(failure within {window:g} h)": p_fail,
            "same at a constant rate": 1.0 - np.exp(-rate_now * window),
        }
    )

window_table = pd.DataFrame(window_rows)
rate_table = pd.DataFrame(rate_columns)
peek("13.window_table", window_table)
peek("13.rate_table", rate_table)
outside = TIMES[~in_range]
print(
    f"\nthe hazard line covers {DRONE_HAZARD_GRID[0]:.1f}–{DRONE_HAZARD_GRID[-1]:.1f} h; "
    + (
        f"probe times outside it: {outside.tolist()} h"
        if len(outside)
        else "all probe times lie inside it"
    )
)
display(window_table)
rate_table

[ok] cox: 1 - S(a+w)/S(a): shape=(1,), max |diff| = 0.000e+00
[ok] aft: 1 - S(a+w)/S(a): shape=(1,), max |diff| = 0.000e+00
[13.window_table] DataFrame shape=(2, 4) dtype=3 x float64, 1 x str size=8 bytes=202
    columns: model, rate at the age, per 100 h, P(failure within 100 h), same at a constant rate
[13.rate_table] DataFrame shape=(3, 6) dtype=5 x float64, 1 x bool size=18 bytes=255
    columns: t, inside the plotted range, h cox, per hour, h cox, per 100 h, h aft, per hour, h aft, per
    100 h

the hazard line covers 42.4–495.7 h; all probe times lie inside it


,model,"rate at the age, per 100 h",P(failure within 100 h),same at a constant rate
0,cox,0.582358,0.47041,0.44142
1,aft,0.503687,0.352994,0.395701


,t,inside the plotted range,"h cox, per hour","h cox, per 100 h","h aft, per hour","h aft, per 100 h"
0,50,True,0.00644312,0.644312,0.00787901,0.787901
1,193.5,True,0.00582358,0.582358,0.00503687,0.503687
2,293.5,True,0.00589016,0.589016,0.00377782,0.377782


## 14. Conditional curve and remaining life

### 14.1 How `remaining_life` gets its numbers

`remaining_life` evaluates the conditional curve $S(a+u)/S(a)$ on its own grid of `n_grid = 1001` points over $[0, \mathrm{horizon}]$ and reads four numbers off it:

- `rul_q10`: the first $u$ at which the curve is at or below 0.9, that is, the failure probability has reached 10%;
- `rul_median`: the first $u$ at which the curve is at or below 0.5;
- `rmrl`: the area under the curve over $[0, \mathrm{horizon}]$ by the trapezoid rule, the mean residual life restricted to the horizon;
- `p_fail_window`: $1 - S(a+w)/S(a)$ evaluated exactly at $w$ = `window`; it does not depend on the grid.

"First" is found with a boolean mask `curve <= level` of shape `(n, 1001)` and `argmax` along the time axis, which returns the index of the first `True`. If the level is not reached within the horizon the result is `inf`. The resolution of `rul_q10` and `rul_median` is one grid step, `horizon / 1000`.

The numbers of the dashboard come from the production model named by `forecast_name` in `meta.json`; it need not be the AFT model of the main walk, so this section takes `MODELS[SETTINGS.forecast]`.

For test drones in service `predictions.parquet` stores the same four numbers; the cell compares them when the traced drone is one of those.

In [34]:
FORECAST_MODEL = MODELS[SETTINGS.forecast]
life = remaining_life(FORECAST_MODEL, drone, SETTINGS.horizon, SETTINGS.window)
peek("14.life", life)

N_LIFE_GRID = 1001
life_grid = np.linspace(0.0, SETTINGS.horizon, N_LIFE_GRID)
life_curve = FORECAST_MODEL.conditional_survival(drone, ages, life_grid)
reached_q10 = life_curve <= 0.9
reached_median = life_curve <= 0.5
first_q10 = reached_q10.argmax(axis=1)
first_median = reached_median.argmax(axis=1)
life_hand = {
    "rul_q10": np.where(reached_q10.any(axis=1), life_grid[first_q10], np.inf),
    "rul_median": np.where(reached_median.any(axis=1), life_grid[first_median], np.inf),
    "rmrl": np.trapezoid(life_curve, life_grid, axis=1),
    "p_fail_window": 1.0 - FORECAST_MODEL.conditional_survival(drone, ages, window_arr)[:, 0],
}
peek("14.life_grid", life_grid)
peek("14.life_curve", life_curve)
peek("14.reached_q10", reached_q10)
peek("14.reached_median", reached_median)
peek("14.first_q10", first_q10)
peek("14.first_median", first_median)
for key, value in life_hand.items():
    peek(f"14.life_hand.{key}", value)
    same(f"{key} = remaining_life", value, life[key])

if IN_TEST and bool(pred["in_service"].iloc[DRONE_POS]):
    for key in life_hand:
        same(f"stored {key} = remaining_life", pred[key].iloc[[DRONE_POS]], life[key], rtol=1e-9)
else:
    print(f"{DRONE_ID} is not a test drone in service: predictions.parquet has no remaining life for it")

print(
    f"\ngrid step {life_grid[1] - life_grid[0]:.4f} h; "
    f"P(failure within {window:g} h) = {life_hand['p_fail_window'][0]:.4f}; "
    f"10% quantile of the remaining life = {life_hand['rul_q10'][0]:.4f} h; "
    f"median = {life_hand['rul_median'][0]:.4f} h; "
    f"mean restricted to the horizon = {life_hand['rmrl'][0]:.2f} h"
)
life

[14.life] DataFrame shape=(1, 6) dtype=5 x float64, 1 x str size=6 bytes=189
    columns: drone_id, age_h, rul_q10, rul_median, rmrl, p_fail_window
[14.life_grid] ndarray shape=(1001,) dtype=float64 size=1001 bytes=8008 | min=0 max=1111 nan=0 inf=0
    [   0.       1.111    2.222 ... 1108.778 1109.889 1111.   ]
[14.life_curve] ndarray shape=(1, 1001) dtype=float64 size=1001 bytes=8008 | min=0.00842644 max=1 nan=0 inf=0
    [[1.       0.994343 0.987299 ... 0.008426 0.008426 0.008426]]
[14.reached_q10] ndarray shape=(1, 1001) dtype=bool size=1001 bytes=1001 | true=985
    [[False False False ...  True  True  True]]
[14.reached_median] ndarray shape=(1, 1001) dtype=bool size=1001 bytes=1001 | true=904
    [[False False False ...  True  True  True]]
[14.first_q10] ndarray shape=(1,) dtype=int64 size=1 bytes=8 | min=16 max=16
    [16]
[14.first_median] ndarray shape=(1,) dtype=int64 size=1 bytes=8 | min=97 max=97
    [97]
[14.life_hand.rul_q10] ndarray shape=(1,) dtype=float64 size=1 bytes=

,drone_id,age_h,rul_q10,rul_median,rmrl,p_fail_window
0,DR-003057,193.5,17.776,107.767,178.111,0.47041


### 14.2 The conditional curve on the dashboard

`figures.conditional_curves` draws $S(a+u)/S(a)$ on `rul_grid`: 300 points from 0 to `rul_horizon`. The x axis is **additional** flight hours $u$, not total hours. The horizontal line at 0.5 marks the median level; it is a layout shape, not an estimate from data. Its x runs from 0 to 1 in axis fractions (`x domain`).

The drone card draws this curve under the trace name "Current conditions". It takes the row through `common.whatif_base`, which restarts the age from 0 for a drone that is no longer in service; this cell always uses `AGE`.

The figure has 300 nodes and `remaining_life` has 1001, so a median read off the figure can differ from `rul_median` by up to one step of the coarser grid.

In [35]:
cond_name = "Current conditions"
cond_title = f"Remaining life from {AGE:.0f} flight hours ({MODEL_LABELS[SETTINGS.forecast]})"
cond_fig = conditional_curves(FORECAST_MODEL, drone, ages, SETTINGS.rul_grid, [cond_name], cond_title)
describe_figure("14.cond_fig", cond_fig)

chart_curve = FORECAST_MODEL.conditional_survival(drone, ages, SETTINGS.rul_grid)
peek("14.chart_curve", chart_curve)
drawn = trace_xy(cond_fig, cond_name)
same("trace x = rul_grid", drawn["x"], SETTINGS.rul_grid)
same("trace y = conditional_survival(rul_grid)[0]", drawn["y"], chart_curve[0])
same("curve at u = 0", chart_curve[:, :1], np.ones((len(drone), 1)))

cond_hline = cond_fig.layout.shapes[0]
same(
    "horizontal line at 0.5",
    np.array([cond_hline.y0, cond_hline.y1], dtype=float),
    np.array([0.5, 0.5]),
)

chart_reached = chart_curve[0] <= 0.5
median_chart = (
    float(SETTINGS.rul_grid[int(chart_reached.argmax())]) if chart_reached.any() else np.inf
)
rul_step = float(SETTINGS.rul_grid[1] - SETTINGS.rul_grid[0])
print(
    f"\nmedian read off the figure (300 nodes) = {median_chart:.4f} h; "
    f"remaining_life (1001 nodes) = {life_hand['rul_median'][0]:.4f} h; "
    f"step of the figure grid {rul_step:.4f} h"
)
display(cond_fig)

[14.cond_fig] Figure: 1 traces, 1 shapes, 1 annotations
    title:  'Remaining life from 194 flight hours (Cox)'
    x axis: 'Additional flight hours', range=None, rangemode=None
    y axis: 'S(a + u) / S(a)', range=(0, 1.02), rangemode=None
    height: 400


,trace,name,type,mode,x shape,y shape,x min,x max,y min,y max,NaN in y,line colour,line width,fill,in legend
0,0,Current conditions,scatter,lines,"(300,)","(300,)",0,1111,0.00842644,1,0,#1f77b4,None,None,True


,kind,xref,x0,x1,yref,y0,y1,colour,dash,text
0,shape/line,x domain,0,1,y,0.5,0.5,grey,dot,NaN
1,annotation,x domain,1,1,y,0.5,0.5,NaN,NaN,median


[14.chart_curve] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.00842644 max=1 nan=0 inf=0
    [[1.       0.973341 0.948516 ... 0.008616 0.008616 0.008426]]
[ok] trace x = rul_grid: shape=(300,), max |diff| = 0.000e+00
[ok] trace y = conditional_survival(rul_grid)[0]: shape=(300,), max |diff| = 0.000e+00
[ok] curve at u = 0: shape=(1, 1), max |diff| = 0.000e+00
[ok] horizontal line at 0.5: shape=(2,), max |diff| = 0.000e+00

median read off the figure (300 nodes) = 107.7559 h; remaining_life (1001 nodes) = 107.7670 h; step of the figure grid 3.7157 h


## 15. From one drone to the fleet hazard curve

`figures.hazard` does not draw the row of one drone. It draws `hazard_table`: three curves on `hazard_grid` computed from all test drones, the observed hazard with its interval and one predicted group hazard per model. The figure is a quality check on the test split, so its two model curves come from the **evaluation** models (`EVAL_MODELS`), not from the production ones.

### 15.1 Observed hazard: smoothed Nelson–Aalen

`observed_hazard` uses only `duration_h` and `event`, no model:

1. the distinct failure times $t_j$ and the number of failures $d_j$ at each, shape `(k,)`;
2. the number of drones still under observation just before each failure time, $n_j$, shape `(k,)`;
3. the Nelson–Aalen increments $d_j / n_j$, the share of the drones at risk that fail at $t_j$;
4. the Epanechnikov weights between every grid point and every failure time, shape `(g, k)`;
5. the smoothed hazard $h(t) = \sum_j K_b(t - t_j)\, d_j / n_j$, shape `(g,)`, and its standard error $\sqrt{\sum_j K_b(t - t_j)^2\, d_j / n_j^2}$;
6. the pointwise 95% interval $h \pm 1.96\,\mathrm{se}$, with the lower bound clipped at 0.

In [36]:
fleet_grid = SETTINGS.hazard_grid
bandwidth = SETTINGS.hazard_bandwidth
durations = pred["duration_h"].to_numpy(dtype=float)
events = pred["event"].to_numpy(dtype=bool)

obs_times, obs_deaths = np.unique(durations[events], return_counts=True)
obs_at_risk = len(durations) - np.searchsorted(np.sort(durations), obs_times, side="left")
obs_increments = obs_deaths / obs_at_risk
obs_weights = _epanechnikov(fleet_grid, obs_times, bandwidth)
obs_h_hand = obs_weights @ obs_increments
obs_se = np.sqrt(obs_weights**2 @ (obs_deaths / obs_at_risk**2))
obs_low_hand = np.maximum(obs_h_hand - 1.96 * obs_se, 0.0)
obs_high_hand = obs_h_hand + 1.96 * obs_se
obs_h, obs_low, obs_high = observed_hazard(durations, events, fleet_grid, bandwidth)

peek("15.durations", durations)
peek("15.events", events)
peek("15.obs_times", obs_times)
peek("15.obs_deaths", obs_deaths)
peek("15.obs_at_risk", obs_at_risk)
peek("15.obs_increments", obs_increments)
peek("15.obs_weights", obs_weights)
peek("15.obs_se", obs_se)
peek("15.obs_h", obs_h)
peek("15.obs_low", obs_low)
peek("15.obs_high", obs_high)
same("observed hazard", obs_h, obs_h_hand)
same("lower bound", obs_low, obs_low_hand)
same("upper bound", obs_high, obs_high_hand)

print("\nfirst failure times:")
pd.DataFrame(
    {
        "failure time": obs_times,
        "failures d": obs_deaths,
        "at risk n": obs_at_risk,
        "increment d/n": obs_increments,
    }
).head()

[15.durations] ndarray shape=(1001,) dtype=float64 size=1001 bytes=8008 | min=0.18 max=1236.24 nan=0 inf=0
    [ 63.67 159.67 312.4  ...  66.78 227.58   1.03]
[15.events] ndarray shape=(1001,) dtype=bool size=1001 bytes=1001 | true=448
    [ True  True False ...  True False  True]
[15.obs_times] ndarray shape=(442,) dtype=float64 size=442 bytes=3536 | min=0.55 max=1156.68 nan=0 inf=0
    [5.50000e-01 9.20000e-01 9.60000e-01 ... 7.60620e+02 8.05900e+02 1.15668e+03]
[15.obs_deaths] ndarray shape=(442,) dtype=int64 size=442 bytes=3536 | min=1 max=2
    [1 1 1 ... 1 1 1]
[15.obs_at_risk] ndarray shape=(442,) dtype=int64 size=442 bytes=3536 | min=3 max=1000
    [1000  999  998 ...   25   20    3]
[15.obs_increments] ndarray shape=(442,) dtype=float64 size=442 bytes=3536 | min=0.001 max=0.333333 nan=0 inf=0
    [0.001    0.001001 0.001002 ... 0.04     0.05     0.333333]
[15.obs_weights] ndarray shape=(200, 442) dtype=float64 size=88400 bytes=707200 | min=0 max=0.0176723 nan=0 inf=0
    [[0.0

,failure time,failures d,at risk n,increment d/n
0,0.55,1,1000,0.001
1,0.92,1,999,0.001001
2,0.96,1,998,0.001002
3,1.03,2,997,0.00200602
4,1.2,1,995,0.00100503


### 15.2 Predicted group hazard: survival-weighted mean

`population_hazard` evaluates a model on every test drone and every grid point: `survival` and `hazard` both return `(n, g)`. Drones that the model expects to have failed by $t$ must leave the average, as failed drones leave the observed risk set, so each drone is weighted by its survival probability:

$$h_{\mathrm{group}}(t) = \sum_i S_i(t)\, h_i(t) \,/\, \sum_i S_i(t).$$

The sums run over the drone axis (`axis=0`) and leave one value per grid point, `(g,)`. The plain mean of $h_i$ is not what is drawn; the cell keeps it only to show the gap.

If the traced drone is in the test table, its row of the `(n, g)` hazard matrix is one term of the sum, and it equals the single-drone hazard evaluated on this grid.

In [37]:
FLEET_SURV, FLEET_HAZARD = {}, {}
WEIGHTED_SUM, WEIGHT_TOTAL = {}, {}
GROUP_HAZARD, PLAIN_MEAN_HAZARD = {}, {}
for kind, model in EVAL_MODELS.items():
    FLEET_SURV[kind] = model.survival(pred, fleet_grid)
    FLEET_HAZARD[kind] = model.hazard(pred, fleet_grid, bandwidth)
    WEIGHTED_SUM[kind] = (FLEET_SURV[kind] * FLEET_HAZARD[kind]).sum(axis=0)
    WEIGHT_TOTAL[kind] = FLEET_SURV[kind].sum(axis=0)
    GROUP_HAZARD[kind] = WEIGHTED_SUM[kind] / WEIGHT_TOTAL[kind]
    PLAIN_MEAN_HAZARD[kind] = FLEET_HAZARD[kind].mean(axis=0)
    peek(f"15.FLEET_SURV.{kind}", FLEET_SURV[kind])
    peek(f"15.FLEET_HAZARD.{kind}", FLEET_HAZARD[kind])
    peek(f"15.WEIGHTED_SUM.{kind}", WEIGHTED_SUM[kind])
    peek(f"15.WEIGHT_TOTAL.{kind}", WEIGHT_TOTAL[kind])
    peek(f"15.GROUP_HAZARD.{kind}", GROUP_HAZARD[kind])
    peek(f"15.PLAIN_MEAN_HAZARD.{kind}", PLAIN_MEAN_HAZARD[kind])
    same(
        f"{kind}: sum(S h) / sum(S) = population_hazard",
        GROUP_HAZARD[kind],
        population_hazard(model, pred, fleet_grid, bandwidth),
    )
    if IN_TEST:
        same(
            f"{kind}: row {DRONE_POS} of the hazard matrix = hazard(drone)",
            FLEET_HAZARD[kind][DRONE_POS],
            model.hazard(drone, fleet_grid, bandwidth)[0],
        )

[15.FLEET_SURV.cox] ndarray shape=(1001, 200) dtype=float64 size=200200 bytes=1601600 | min=2.13113e-06 max=0.983396 nan=0 inf=0
    [[0.968453 0.966704 0.965286 ... 0.650272 0.64988  0.649095]
     [0.966053 0.964175 0.962652 ... 0.630216 0.62981  0.628998]
     [0.945451 0.942477 0.940069 ... 0.485063 0.484591 0.483648]
     ...
     [0.93627  0.932819 0.930026 ... 0.433331 0.432852 0.431893]
     [0.977226 0.975956 0.974926 ... 0.730363 0.730037 0.729382]
     [0.822317 0.813519 0.806457 ... 0.124161 0.123872 0.123295]]
[15.FLEET_HAZARD.cox] ndarray shape=(1001, 200) dtype=float64 size=200200 bytes=1601600 | min=0.000350478 max=0.055364 nan=0 inf=0
    [[0.000682 0.000677 0.000673 ... 0.000864 0.000865 0.00087 ]
     [0.000735 0.000729 0.000725 ... 0.000923 0.000924 0.000929]
     [0.001189 0.00118  0.001172 ... 0.001394 0.001395 0.001403]
     ...
     [0.001394 0.001383 0.001374 ... 0.001586 0.001587 0.001596]
     [0.000491 0.000487 0.000484 ... 0.000641 0.000642 0.000646]
     [

One grid point makes the weighting concrete. The cell takes the node of `hazard_grid` nearest to the drone's age and shows, for each model, the weighted and the plain mean, the total weight $\sum_i S_i$ and, if the drone is in the test table, its share of that weight and its own hazard.

In [38]:
point_i, t_star = nearest(fleet_grid)
point_rows = []
for kind in EVAL_MODELS:
    surv_col = FLEET_SURV[kind][:, point_i]
    rate_col = FLEET_HAZARD[kind][:, point_i]
    point_row = {
        "source": kind,
        "h group, per hour": GROUP_HAZARD[kind][point_i],
        f"h group, per {HAZARD_SCALE:g} h": HAZARD_SCALE * GROUP_HAZARD[kind][point_i],
        "plain mean of h, per hour": PLAIN_MEAN_HAZARD[kind][point_i],
        "sum of S": WEIGHT_TOTAL[kind][point_i],
    }
    if IN_TEST:
        point_row["weight share of the drone"] = surv_col[DRONE_POS] / surv_col.sum()
        point_row["h of the drone, per hour"] = rate_col[DRONE_POS]
    point_rows.append(point_row)

point_table = pd.DataFrame(point_rows)
peek("15.point_table", point_table)
print(
    f"\nnode of hazard_grid nearest to the age: {t_star:.2f} h (index {point_i}); "
    f"observed rate {HAZARD_SCALE * obs_h[point_i]:.4f} per {HAZARD_SCALE:g} h, "
    f"95% interval [{HAZARD_SCALE * obs_low[point_i]:.4f}, {HAZARD_SCALE * obs_high[point_i]:.4f}]"
)
point_table

[15.point_table] DataFrame shape=(2, 7) dtype=6 x float64, 1 x str size=14 bytes=250
    columns: source, h group, per hour, h group, per 100 h, plain mean of h, per hour, sum of S, weight
    share of the drone, h of the drone, per hour

node of hazard_grid nearest to the age: 192.77 h (index 66); observed rate 0.1646 per 100 h, 95% interval [0.1248, 0.2045]


,source,"h group, per hour","h group, per 100 h","plain mean of h, per hour",sum of S,weight share of the drone,"h of the drone, per hour"
0,cox,0.00156395,0.156395,0.00202334,716.426,0.000362402,0.00636831
1,aft,0.00178027,0.178027,0.00221337,685.724,0.000365319,0.00526702


### 15.3 The table and the figure

`hazard_table` stacks the three curves in long format: `g` rows per source (`observed`, `cox`, `aft`) with the columns `t` and `h`, per flight hour. `ci_low` and `ci_high` are filled for the observed rows only.

`figures.hazard` multiplies everything by `HAZARD_SCALE = 100`, so the y axis is failures per 100 flight hours. Called with the list of models (`kinds`), it adds five traces in this order:

1. the upper bound of the interval, an invisible line of width 0;
2. the lower bound, filled up to the previous trace (`fill="tonexty"`); this pair makes the grey band;
3. the observed line;
4. and 5. one line per model.

The two band traces have no name and are hidden from the legend. The grey band is the 95% interval of the **observed** hazard, not a spread between the models. The dashboard also draws the figure for the selected model only, which leaves four traces.

In [39]:
table = hazard_table(DATA, SETTINGS)
peek("15.hazard_table", table)
print(f"rows per source: {table['source'].value_counts().to_dict()}")

observed = table[table["source"] == "observed"]
same("table: observed h", observed["h"], obs_h)
same("table: ci_low", observed["ci_low"], obs_low)
same("table: ci_high", observed["ci_high"], obs_high)
for kind in EVAL_MODELS:
    same(f"table: h of {kind} = group hazard", table.loc[table["source"] == kind, "h"], GROUP_HAZARD[kind])

fleet_fig = fleet_hazard(table, list(EVAL_MODELS))
describe_figure("15.fleet_fig", fleet_fig)

band = [i for i, tr in enumerate(fleet_fig.data) if not tr.name]
if len(band) != 2:
    raise RuntimeError(f"the band has {len(band)} unnamed traces, expected two")
same(
    f"upper edge of the band = {HAZARD_SCALE:g} * ci_high",
    trace_xy(fleet_fig, band[0])["y"],
    HAZARD_SCALE * obs_high,
)
same(
    f"lower edge of the band = {HAZARD_SCALE:g} * ci_low",
    trace_xy(fleet_fig, band[1])["y"],
    HAZARD_SCALE * obs_low,
)
same(
    f"observed line = {HAZARD_SCALE:g} * h",
    trace_xy(fleet_fig, 2)["y"],
    HAZARD_SCALE * obs_h,
)
for kind in EVAL_MODELS:
    drawn = trace_xy(fleet_fig, f"{MODEL_LABELS[kind]}: prediction")
    same(f"{kind} line: x = hazard_grid", drawn["x"], fleet_grid)
    same(f"{kind} line: y = {HAZARD_SCALE:g} * group hazard", drawn["y"], HAZARD_SCALE * GROUP_HAZARD[kind])

display(fleet_fig)
print("first rows of each source:")
table.groupby("source", sort=False).head(2)

[15.hazard_table] DataFrame shape=(600, 5) dtype=4 x float64, 1 x str size=3000 bytes=27007
    columns: source, t, h, ci_low, ci_high
rows per source: {'observed': 200, 'cox': 200, 'aft': 200}
[ok] table: observed h: shape=(200,), max |diff| = 0.000e+00
[ok] table: ci_low: shape=(200,), max |diff| = 0.000e+00
[ok] table: ci_high: shape=(200,), max |diff| = 0.000e+00
[ok] table: h of cox = group hazard: shape=(200,), max |diff| = 0.000e+00
[ok] table: h of aft = group hazard: shape=(200,), max |diff| = 0.000e+00
[15.fleet_fig] Figure: 5 traces, 0 shapes, 0 annotations
    title:  'Failure rate: predicted vs observed'
    x axis: 'Flight hours', range=None, rangemode=None
    y axis: 'Failures per 100 flight hours', range=None, rangemode=tozero
    height: 400


,trace,name,type,mode,x shape,y shape,x min,x max,y min,y max,NaN in y,line colour,line width,fill,in legend
0,0,NaN,scatter,lines,"(200,)","(200,)",42.4393,495.71,0.182687,0.309557,0,NaN,0,NaN,False
1,1,NaN,scatter,lines,"(200,)","(200,)",42.4393,495.71,0.0383456,0.16041,0,NaN,0,tonexty,False
2,2,Kaplan-Meier (observed),scatter,lines,"(200,)","(200,)",42.4393,495.71,0.121227,0.228797,0,#222222,NaN,NaN,True
3,3,Cox: prediction,scatter,lines,"(200,)","(200,)",42.4393,495.71,0.129324,0.182534,0,#1f77b4,NaN,NaN,True
4,4,AFT: prediction,scatter,lines,"(200,)","(200,)",42.4393,495.71,0.12594,0.212727,0,#ff7f0e,NaN,NaN,True


[ok] upper edge of the band = 100 * ci_high: shape=(200,), max |diff| = 0.000e+00
[ok] lower edge of the band = 100 * ci_low: shape=(200,), max |diff| = 0.000e+00
[ok] observed line = 100 * h: shape=(200,), max |diff| = 0.000e+00
[ok] cox line: x = hazard_grid: shape=(200,), max |diff| = 0.000e+00
[ok] cox line: y = 100 * group hazard: shape=(200,), max |diff| = 0.000e+00
[ok] aft line: x = hazard_grid: shape=(200,), max |diff| = 0.000e+00
[ok] aft line: y = 100 * group hazard: shape=(200,), max |diff| = 0.000e+00


first rows of each source:


,source,t,h,ci_low,ci_high
0,observed,42.4393,0.00193579,0.00159572,0.00227586
1,observed,44.717,0.00190809,0.00156975,0.00224642
200,cox,42.4393,0.00176073,NaN,NaN
201,cox,44.717,0.00173427,NaN,NaN
400,aft,42.4393,0.00212408,NaN,NaN
401,aft,44.717,0.002126,NaN,NaN


## 16. Index of captured intermediates

Every object passed to `peek` or `describe_figure` is in `TRACE`. The table lists them in the order of the walk with type, shape, dtype, number of elements and memory in bytes.

To look at any of them, index the dictionary in a new cell, for example `TRACE["4.z"]`, `TRACE["9.kernel"].shape` or `TRACE["15.fleet_fig"].data[2].y`. The top-level variables of the cells above are available under their own names as well.

In [40]:
captured = trace_index()
print(f"{len(captured)} objects captured")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 60):
    display(captured)

150 objects captured


,name,type,shape,dtype,size,bytes
0,1.META,dict,None,NaN,21,<NA>
1,1.MODEL.params,dict,None,NaN,8,<NA>
2,1.MODEL.features,list,None,NaN,16,<NA>
3,1.MODEL.cat_features,list,None,NaN,4,<NA>
4,1.MODEL.categories,dict,None,NaN,3,<NA>
5,1.MODEL.calibration,NoneType,None,NaN,<NA>,<NA>
6,1.MODEL._breslow,NoneType,None,NaN,<NA>,<NA>
7,1.MODEL.history_lengths,dict,None,NaN,1,<NA>
8,1.fleet,DataFrame,"(5000, 21)","11 x float64, 5 x str, 4 x int64, 1 x bool",105000,1021733
9,1.drone,DataFrame,"(1, 21)","11 x float64, 5 x str, 4 x int64, 1 x bool",21,340


### Where to look first

- **A margin that differs from your own `booster.predict`**: section 2.3 (`output_margin`, `iteration_range`) and section 2.2 (feature order and types).
- **A surprising prediction for a new drone**: section 2.1 (an unseen category becomes NaN) and section 2.5 (which features move the margin).
- **A value that differs between this notebook and the dashboard**: sections 11–13 (a curve is evaluated at grid nodes, not at the exact age, and the Cox hazard depends on the bandwidth).
- **A Cox curve that flattens out**: section 9.5 (the baseline ends at the last training failure).
- **Stored predictions or metrics that no longer match**: section 10 (the artifacts and the model files come from different runs).